<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/DL-2026/Lecture_4/Lecture_4.%20Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Введение в Трансформеры: Революция в обработке последовательностей

За последние несколько лет мир искусственного интеллекта изменился до неузнаваемости. Модели, которые еще вчера казались фантастикой — от генераторов текста (ChatGPT, LLaMA) до систем перевода и анализа кода — сегодня стали обыденностью. Все они, без исключения, построены на одной архитектуре — **Трансформере (Transformer)**.

В предыдущих частях нашего курса мы подробно разобрали полносвязные сети (MLP), сверточные сети (CNN) и рекуррентные сети (RNN). Каждая из них совершила свою революцию: CNN научились видеть картинки, RNN — обрабатывать текст и временные ряды. Но у них был один общий недостаток: они плохо справлялись с **длинными последовательностями** и **глобальным контекстом**.

## Почему RNN и CNN проиграли?

Представьте, что вы переводите длинное предложение.
*   **Рекуррентные сети (RNN)** читают текст строго слева направо, слово за словом. Они хранят в памяти скрытое состояние \(h_t = f(h_{t-1}, x_t)\), которое должно помнить всё, что было раньше. Но чем длиннее предложение, тем сложнее сети не забыть начало. Это приводит к **затуханию градиента**. Кроме того, RNN нельзя распараллелить: чтобы обработать 100-е слово, нужно сначала обработать 99 предыдущих.
*   **Сверточные сети (CNN)** смотрят на локальные окна (например, по 3 слова). Чтобы связать первое и последнее слово предложения, нужно наложить десятки слоев друг на друга, что тоже неэффективно.

## Прорыв: «Внимание — это всё, что вам нужно»

В 2017 году исследователи из Google опубликовали статью, которая изменила всё. Они предложили механизм **Attention (Внимание)** и архитектуру Трансформер.

Главная идея звучит гениально просто: **каждое слово в предложении может напрямую «посмотреть» на любое другое слово, независимо от того, насколько далеко оно находится**. Больше не нужно передавать информацию последовательно через скрытые состояния. Сеть сразу видит весь контекст.

Математически это выражается через скалярное произведение векторов **Запросов (Queries)**, **Ключей (Keys)** и **Значений (Values)**. Благодаря этому:
1.  **Глобальный контекст:** Сеть понимает связи между словами на любом расстоянии.
2.  **Параллелизм:** Все слова обрабатываются одновременно (что идеально для GPU).
3.  **Отсутствие рекуррентности:** Исчезает проблема затухания градиента на длинных дистанциях.

## О чем этот раздел?

В этом разделе  мы шаг за шагом разберем, как устроен Трансформер изнутри. Мы не будем использовать готовые библиотеки как «черный ящик». Вместо этого мы повторим путь исследователей и выведем каждую формулу с нуля.




# Глава 1. Механизм Attention (Внимание): один Query ищет информацию

В предыдущих частях мы изучали сети, которые обрабатывают данные последовательно (RNN) или локально (CNN). Трансформеры обрабатывают данные **глобально**. Главный инструмент для этого — механизм **Attention (Внимание)**.

Представьте, что вы ищете информацию в библиотеке. У вас есть ваш вопрос (Query). На полках стоят книги. У каждой книги есть название на корешке (Key) и само содержимое (Value). Вы сравниваете свой вопрос с названиями книг, выбираете те, что подходят лучше всего, и берете их содержимое.

Именно так работает Attention.

## 1.1. Зачем нужен этот пример

Полноценный Self-Attention работает с матрицами, но в основе лежит скалярное произведение. Чтобы понять, откуда берутся градиенты и как работает Softmax, мы возьмем **один вектор Query** и **три пары Key-Value**.

В этом примере у нас есть:
- **Query** \(q\) (вектор длины \(d=2\)) — то, что мы ищем.
- **Keys** \(k_1, k_2, k_3\) (векторы длины \(d=2\)) — описания доступных данных.
- **Values** \(v_1, v_2, v_3\) (векторы длины \(d_v=2\)) — сами данные.

*Важное примечание:* В реальных Трансформерах \(q, k, v\) не задаются вручную, а получаются из входных данных через обучаемые матрицы \(W_Q, W_K, W_V\). Но чтобы не усложнять первую главу, мы представим, что \(q, k, v\) — это **обучаемые параметры** (как веса в свёртке).

## 1.2. Архитектура и обозначения (Рис. 1.1)

| Символ | Смысл |
|---|---|
| \(q\) | Query (запрос), вектор длины \(d=2\) |
| \(k_i\) | Key (ключ) \(i\)-го элемента, вектор длины \(d=2\) |
| \(v_i\) | Value (значение) \(i\)-го элемента, вектор длины \(d_v=2\) |
| \(s_i\) | Сырая оценка внимания (score): \(q \cdot k_i\) |
| \(\hat{s}_i\) | Масштабированная оценка: \(s_i / \sqrt{d}\) |
| \(\alpha_i\) | Вес внимания (после Softmax) |
| \(\text{out}\) | Итоговый выход (взвешенная сумма \(v_i\)) |
| \(t\) | Истинный ответ (целевой вектор) |
| \(L\) | Функция потерь (MSE) |
| \(\eta\) | Шаг обучения |

```python
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyArrowPatch

def draw_attention_architecture():
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.set_xlim(0, 16); ax.set_ylim(0, 8)
    ax.axis('off')

    # Query
    ax.add_patch(patches.Rectangle((1, 4), 1, 2, facecolor='#AEDFF7', edgecolor='black', lw=1.5))
    ax.text(1.5, 5.0, '$q$', ha='center', va='center', fontsize=18)
    ax.text(1.5, 3.5, 'Query', ha='center', fontsize=12, style='italic')

    # Keys
    for i in range(3):
        y_pos = 6.5 - i * 2.5
        ax.add_patch(patches.Rectangle((4, y_pos), 1, 1, facecolor='#FFE29A', edgecolor='black', lw=1.5))
        ax.text(4.5, y_pos + 0.5, f'$k_{i+1}$', ha='center', va='center', fontsize=14)
    ax.text(4.5, 7.5, 'Keys (Ключи)', ha='center', fontsize=12, style='italic')

    # Scores и Softmax
    for i in range(3):
        y_pos = 6.5 - i * 2.5
        ax.add_patch(patches.Circle((7.5, y_pos + 0.5), 0.4, facecolor='#E8E8E8', edgecolor='black'))
        ax.text(7.5, y_pos + 0.5, f'$\\alpha_{i+1}$', ha='center', va='center', fontsize=12)
    ax.text(7.5, 7.5, 'Вес (Softmax)', ha='center', fontsize=12, style='italic')

    # Values
    for i in range(3):
        y_pos = 6.5 - i * 2.5
        ax.add_patch(patches.Rectangle((10, y_pos), 1, 1, facecolor='#B8E0B8', edgecolor='black', lw=1.5))
        ax.text(10.5, y_pos + 0.5, f'$v_{i+1}$', ha='center', va='center', fontsize=14)
    ax.text(10.5, 7.5, 'Values (Значения)', ha='center', fontsize=12, style='italic')

    # Выход
    ax.add_patch(patches.Rectangle((13, 4), 1.5, 2, facecolor='#F5B7B1', edgecolor='black', lw=1.5))
    ax.text(13.75, 5.0, 'out', ha='center', va='center', fontsize=14)
    ax.text(13.75, 3.5, 'Выход', ha='center', fontsize=12, style='italic')

    # Стрелки
    for i in range(3):
        y_pos = 6.5 - i * 2.5
        ax.annotate('', xy=(4.0, y_pos + 0.5), xytext=(2.0, 5.0), arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=1.5))
        ax.annotate('', xy=(7.1, y_pos + 0.5), xytext=(5.0, y_pos + 0.5), arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=1.5))
        ax.annotate('', xy=(10.0, y_pos + 0.5), xytext=(7.9, y_pos + 0.5), arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=1.5))
        ax.annotate('', xy=(13.0, 5.0), xytext=(11.0, y_pos + 0.5), arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=1.5))

    ax.text(8.0, 0.5, 'Рис. 1.1. Механизм Attention: Query сравнивается с Keys, получает веса и агрегирует Values.',
            ha='center', fontsize=13, fontweight='bold')

    plt.tight_layout(); plt.show()

draw_attention_architecture()
```

## 1.3. Прямой проход (Forward Pass)

### Шаг 1. Оценка схожести (Score)
Мы сравниваем Query с каждым Key с помощью скалярного произведения. Чем ближе векторы, тем больше скалярное произведение.
$$
s_i = q \cdot k_i = \sum_{m=1}^{d} q_m \cdot k_{i,m}, \quad i=1,2,3.
$$
Для нашего примера (\(d=2\)):
$$
s_1 = q_1 k_{1,1} + q_2 k_{1,2},
$$
$$
s_2 = q_1 k_{2,1} + q_2 k_{2,2},
$$
$$
s_3 = q_1 k_{3,1} + q_2 k_{3,2}.
$$

### Шаг 2. Масштабирование (Scale)
Если размерность \(d\) большая, скалярные произведения могут быть очень большими. Это приведет к тому, что Softmax «насытится» (одна вероятность станет 1, остальные 0), и градиенты исчезнут. Чтобы этого избежать, делим на корень из размерности:
$$
\hat{s}_i = \frac{s_i}{\sqrt{d}}.
$$
Для \(d=2\), \(\sqrt{d} \approx 1.4142\).

### Шаг 3. Softmax (Веса внимания)
Превращаем оценки в вероятности (сумма = 1):
$$
\alpha_i = \text{softmax}(\hat{s})_i = \frac{e^{\hat{s}_i}}{\sum_{j=1}^{3} e^{\hat{s}_j}}.
$$

### Шаг 4. Взвешенная сумма (Выход)
Итоговый вектор — это сумма значений \(v_i\), взвешенных по \(\alpha_i\):
$$
\text{out} = \sum_{i=1}^{3} \alpha_i v_i.
$$

## 1.4. Функция потерь

Пусть у нас есть целевой вектор \(t\) (размерности \(d_v=2\)). Мы хотим, чтобы выход сети \(\text{out}\) совпал с \(t\). Используем MSE:
$$
L = \frac{1}{2} \sum_{m=1}^{d_v} (\text{out}_m - t_m)^2.
$$




### 1.5. Обратный проход (Backpropagation) — Полный вывод

Нам нужно найти производные функции потерь \(L\) по всем параметрам: \(q, k_1, k_2, k_3, v_1, v_2, v_3\).
Мы будем двигаться строго **справа налево** по графу вычислений: от потерь \(L\) к выходу \(\text{out}\), затем через Softmax к \(\alpha_i\), затем к оценкам \(s_i\), и наконец к векторам \(q\) и \(k_i\).

#### Шаг 1. Градиент по выходу (out)
Функция потерь: \(L = \frac{1}{2} \sum_{m=1}^{d_v} (\text{out}_m - t_m)^2\).
Дифференцируем по каждому элементу выхода:
$$
\frac{\partial L}{\partial \text{out}_m} = \text{out}_m - t_m.
$$
Обозначим вектор ошибки как \(d\text{out} = \text{out} - t\). (Его размерность \(d_v\)).

#### Шаг 2. Градиенты по Values (\(v_i\))
Выход формируется как взвешенная сумма: \(\text{out}_m = \sum_{j=1}^{3} \alpha_j v_{j,m}\).
Найдем производную по \(i\)-му значению \(v_i\) (для каждого \(i=1,2,3\)). Применяем цепное правило:
$$
\frac{\partial L}{\partial v_{i,m}} = \sum_{j=1}^{d_v} \frac{\partial L}{\partial \text{out}_j} \cdot \frac{\partial \text{out}_j}{\partial v_{i,m}}.
$$
Поскольку \(\text{out}_j\) зависит только от \(v_{i,j}\) (а не от других компонент), производная \(\frac{\partial \text{out}_j}{\partial v_{i,m}}\) равна \(\alpha_i\), если \(j=m\), и \(0\) иначе. Сумма схлопывается:
$$
\frac{\partial L}{\partial v_{i,m}} = \frac{\partial L}{\partial \text{out}_m} \cdot \alpha_i.
$$
В векторном виде:
$$
dv_i = \alpha_i \cdot d\text{out}.
$$

#### Шаг 3. Градиенты по весам внимания (\(\alpha_i\))
Нам нужно найти \(\frac{\partial L}{\partial \alpha_i}\).
$$
\frac{\partial L}{\partial \alpha_i} = \sum_{m=1}^{d_v} \frac{\partial L}{\partial \text{out}_m} \cdot \frac{\partial \text{out}_m}{\partial \alpha_i}.
$$
Из формулы \(\text{out}_m = \sum_j \alpha_j v_{j,m}\) видно, что \(\frac{\partial \text{out}_m}{\partial \alpha_i} = v_{i,m}\). Подставляем:
$$
\frac{\partial L}{\partial \alpha_i} = \sum_{m=1}^{d_v} d\text{out}_m \cdot v_{i,m} = v_i \cdot d\text{out}.
$$
Это скалярное произведение вектора \(v_i\) и вектора ошибки. Обозначим этот скаляр как \(d\alpha_i\).

#### Шаг 4. Градиенты через Softmax (\(d\hat{s}_i\))
Это самый сложный шаг. Веса \(\alpha\) получаются через Softmax от оценок \(\hat{s}\):
$$
\alpha_i = \frac{e^{\hat{s}_i}}{\sum_{j=1}^{3} e^{\hat{s}_j}}.
$$
Нам нужно найти \(\frac{\partial L}{\partial \hat{s}_k}\). Применяем цепное правило по всем \(i\):
$$
\frac{\partial L}{\partial \hat{s}_k} = \sum_{i=1}^{3} \frac{\partial L}{\partial \alpha_i} \cdot \frac{\partial \alpha_i}{\partial \hat{s}_k}.
$$
Нам нужна производная Softmax \(\frac{\partial \alpha_i}{\partial \hat{s}_k}\). Она имеет два случая (из-за того, что знаменатель тоже зависит от \(\hat{s}_k\)):
- Если \(i = k\): \(\frac{\partial \alpha_k}{\partial \hat{s}_k} = \alpha_k(1 - \alpha_k)\).
- Если \(i \neq k\): \(\frac{\partial \alpha_i}{\partial \hat{s}_k} = -\alpha_i \alpha_k\).

Подставляем эти два случая в сумму:
$$
\frac{\partial L}{\partial \hat{s}_k} = \frac{\partial L}{\partial \alpha_k} \alpha_k(1 - \alpha_k) + \sum_{i \neq k} \frac{\partial L}{\partial \alpha_i} (-\alpha_i \alpha_k).
$$
Раскроем скобки и вынесем общий множитель \(-\alpha_k\):
$$
\frac{\partial L}{\partial \hat{s}_k} = \frac{\partial L}{\partial \alpha_k} \alpha_k - \frac{\partial L}{\partial \alpha_k} \alpha_k^2 - \alpha_k \sum_{i \neq k} \frac{\partial L}{\partial \alpha_i} \alpha_i.
$$
Сгруппируем первое слагаемое и добавим/вычтем \(\alpha_k \frac{\partial L}{\partial \alpha_k} \alpha_k\), чтобы получить полную сумму:
$$
\frac{\partial L}{\partial \hat{s}_k} = \alpha_k \left( \frac{\partial L}{\partial \alpha_k} - \sum_{i=1}^{3} \frac{\partial L}{\partial \alpha_i} \alpha_i \right).
$$
Обозначим \(d\alpha_i = \frac{\partial L}{\partial \alpha_i}\). Итоговая формула:
$$
d\hat{s}_k = \alpha_k \left( d\alpha_k - \sum_{i=1}^{3} \alpha_i \cdot d\alpha_i \right).
$$
*(Смысл: мы вычитаем «средневзвешенный» градиент, чтобы общая сумма весов оставалась равной 1).*

#### Шаг 5. Градиенты через масштабирование (\(ds_i\))
Оценки масштабировались как \(\hat{s}_i = s_i / \sqrt{d}\). Производная — это просто умножение на константу:
$$
ds_i = \frac{d\hat{s}_i}{\sqrt{d}}.
$$

#### Шаг 6. Градиент по Query (\(q\))
Оценка \(s_i\) — это скалярное произведение \(q\) и \(k_i\):
$$
s_i = \sum_{m=1}^{d} q_m \cdot k_{i,m}.
$$
Нам нужно найти производную по \(q_m\) (компоненте вектора \(q\)). Применяем цепное правило по всем \(i\):
$$
\frac{\partial L}{\partial q_m} = \sum_{i=1}^{3} \frac{\partial L}{\partial s_i} \cdot \frac{\partial s_i}{\partial q_m}.
$$
Из формулы видно, что \(\frac{\partial s_i}{\partial q_m} = k_{i,m}\). Подставляем:
$$
\frac{\partial L}{\partial q_m} = \sum_{i=1}^{3} ds_i \cdot k_{i,m}.
$$
В векторном виде:
$$
dq = \sum_{i=1}^{3} ds_i \cdot k_i.
$$

#### Шаг 7. Градиенты по Keys (\(k_i\))
Аналогично, производная \(s_i\) по компоненте \(k_{i,m}\):
$$
\frac{\partial L}{\partial k_{i,m}} = \frac{\partial L}{\partial s_i} \cdot \frac{\partial s_i}{\partial k_{i,m}}.
$$
Поскольку \(s_i = \sum_m q_m k_{i,m}\), производная по \(k_{i,m}\) равна \(q_m\). Следовательно:
$$
\frac{\partial L}{\partial k_{i,m}} = ds_i \cdot q_m.
$$
В векторном виде (для каждого \(i\)):
$$
dk_i = ds_i \cdot q.
$$

---

### 1.7. Числовой пример (с детальным обратным проходом)

Дано:
- \(q = (1, 0)\)
- \(k_1 = (1, 1)\), \(k_2 = (0, 2)\), \(k_3 = (-1, 1)\)
- \(v_1 = (1, 2)\), \(v_2 = (3, 0)\), \(v_3 = (0, 1)\)
- Цель \(t = (1, 1)\)

#### Прямой проход (напоминание)
- \(s = (1, 0, -1)\)
- \(\hat{s} = s / \sqrt{2} \approx (0.707, 0, -0.707)\)
- \(\alpha \approx (0.576, 0.284, 0.140)\)
- \(\text{out} = (1.428, 1.292)\)
- Потеря \(L \approx 0.134\).

#### Обратный проход (шаг за шагом)

**Шаг 1.** Вектор ошибки:
$$
d\text{out} = \text{out} - t = (1.428 - 1, 1.292 - 1) = (0.428, 0.292).
$$

**Шаг 2.** Градиенты по Values:
$$
dv_1 = 0.576 \cdot (0.428, 0.292) = (0.246, 0.168).
$$
$$
dv_2 = 0.284 \cdot (0.428, 0.292) = (0.122, 0.083).
$$
$$
dv_3 = 0.140 \cdot (0.428, 0.292) = (0.060, 0.041).
$$

**Шаг 3.** Градиенты по \(\alpha\) (скалярные произведения):
$$
d\alpha_1 = v_1 \cdot d\text{out} = 1(0.428) + 2(0.292) = 1.012.
$$
$$
d\alpha_2 = v_2 \cdot d\text{out} = 3(0.428) + 0(0.292) = 1.284.
$$
$$
d\alpha_3 = v_3 \cdot d\text{out} = 0(0.428) + 1(0.292) = 0.292.
$$

**Шаг 4.** Проход через Softmax:
Вычисляем сумму: \(S = \sum \alpha_i d\alpha_i = 0.576(1.012) + 0.284(1.284) + 0.140(0.292) \approx 0.583 + 0.365 + 0.041 = 0.989\).
Теперь \(d\hat{s}_i = \alpha_i(d\alpha_i - S)\):
$$
d\hat{s}_1 = 0.576 \cdot (1.012 - 0.989) = 0.576(0.023) = 0.0132.
$$
$$
d\hat{s}_2 = 0.284 \cdot (1.284 - 0.989) = 0.284(0.295) = 0.0838.
$$
$$
d\hat{s}_3 = 0.140 \cdot (0.292 - 0.989) = 0.140(-0.697) = -0.0976.
$$

**Шаг 5.** Градиенты по score (\(ds\)):
$$
ds_i = d\hat{s}_i / \sqrt{2} = d\hat{s}_i / 1.414.
$$
$$
ds_1 = 0.0132 / 1.414 \approx 0.0093.
$$
$$
ds_2 = 0.0838 / 1.414 \approx 0.0593.
$$
$$
ds_3 = -0.0976 / 1.414 \approx -0.0690.
$$

**Шаг 6.** Градиент по Query (\(dq\)):
$$
dq = ds_1 k_1 + ds_2 k_2 + ds_3 k_3.
$$
$$
dq = 0.0093(1, 1) + 0.0593(0, 2) - 0.0690(-1, 1).
$$
$$
dq_x = 0.0093 + 0 + 0.0690 = 0.0783.
$$
$$
dq_y = 0.0093 + 0.1186 - 0.0690 = 0.0589.
$$
Итого: \(dq = (0.0783, 0.0589)\).

**Шаг 7.** Градиенты по Keys (\(dk\)):
$$
dk_1 = ds_1 \cdot q = 0.0093 \cdot (1, 0) = (0.0093, 0).
$$
$$
dk_2 = ds_2 \cdot q = 0.0593 \cdot (1, 0) = (0.0593, 0).
$$
$$
dk_3 = ds_3 \cdot q = -0.0690 \cdot (1, 0) = (-0.0690, 0).
$$

#### Обновление (шаг \(\eta = 0.1\))
$$
q \leftarrow (1, 0) - 0.1(0.0783, 0.0589) = (0.99217, -0.00589),
$$
$$
k_1 \leftarrow (1, 1) - 0.1(0.0093, 0) = (0.99907, 1),
$$
$$
v_1 \leftarrow (1, 2) - 0.1(0.246, 0.168) = (0.9754, 1.9832),
$$
и так далее для остальных параметров.

---

### Код с полной проверкой

Теперь код в точности повторяет все шаги, включая промежуточные переменные (чтобы можно было напечатать и сверить).

```python
import numpy as np

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum()

class Attention1D:
    def __init__(self, q, K, V):
        self.q = np.array(q, dtype=float)
        self.K = np.array(K, dtype=float)
        self.V = np.array(V, dtype=float)

    def forward(self):
        d = len(self.q)
        self.s = self.K @ self.q
        self.s_hat = self.s / np.sqrt(d)
        self.alpha = softmax(self.s_hat)
        self.out = self.alpha @ self.V
        return self.out

    def backward(self, t):
        d = len(self.q)
        d_out = self.out - t
        
        # 1. dV
        dV = np.outer(self.alpha, d_out)
        
        # 2. d_alpha
        d_alpha = self.V @ d_out
        
        # 3. Softmax backprop
        sum_term = np.sum(self.alpha * d_alpha)
        d_s_hat = self.alpha * (d_alpha - sum_term)
        
        # 4. Scale backprop
        d_s = d_s_hat / np.sqrt(d)
        
        # 5. dq
        dq = self.K.T @ d_s
        
        # 6. dK
        dK = np.outer(d_s, self.q)
        
        return dq, dK, dV

    def update(self, dq, dK, dV, eta):
        self.q -= eta * dq
        self.K -= eta * dK
        self.V -= eta * dV

if __name__ == '__main__':
    q = [1.0, 0.0]
    K = [[1.0, 1.0], [0.0, 2.0], [-1.0, 1.0]]
    V = [[1.0, 2.0], [3.0, 0.0], [0.0, 1.0]]
    t = np.array([1.0, 1.0])
    eta = 0.1

    net = Attention1D(q, K, V)
    out = net.forward()
    
    print("=== Прямой проход ===")
    print(f"s (raw):    {np.round(net.s, 4)}")
    print(f"s_hat:      {np.round(net.s_hat, 4)}")
    print(f"alpha:      {np.round(net.alpha, 4)}")
    print(f"out:        {np.round(out, 4)}")
    
    dq, dK, dV = net.backward(t)
    
    print("\n=== Обратный проход ===")
    print(f"d_out:      {np.round(out - t, 4)}")
    print(f"dV:\n{np.round(dV, 4)}")
    print(f"d_alpha:    {np.round(net.V @ (out - t), 4)}")
    print(f"dq:         {np.round(dq, 4)}")
    print(f"dK:\n{np.round(dK, 4)}")
    
    net.update(dq, dK, dV, eta)
    print("\n=== После шага ===")
    print(f"q:          {np.round(net.q, 4)}")
    print(f"K:\n{np.round(net.K, 4)}")
    print(f"V:\n{np.round(net.V, 4)}")
```

Теперь каждый шаг обратного прохода имеет строгую формулу, привязанную к конкретному индексу, и полностью проверен численно. Это дает полную картину того, как обучается механизм внимания.



## 1.8. Полная программа

```python
import numpy as np

def softmax(x):
    e_x = np.exp(x - np.max(x)) # вычитаем max для стабильности
    return e_x / e_x.sum()

class Attention1D:
    def __init__(self, q, K, V):
        self.q = np.array(q, dtype=float)
        self.K = np.array(K, dtype=float) # (3, 2)
        self.V = np.array(V, dtype=float) # (3, 2)

    def forward(self):
        d = len(self.q)
        # 1. Scores
        s = self.K @ self.q # (3,)
        # 2. Scale
        s_hat = s / np.sqrt(d)
        # 3. Softmax
        self.alpha = softmax(s_hat)
        # 4. Output
        out = self.alpha @ self.V # (2,)
        return out, s_hat, self.alpha

    def backward(self, out, t):
        d = len(self.q)
        d_out = out - t # (2,)
        
        # Градиенты по V
        dV = np.outer(self.alpha, d_out) # (3, 2)
        
        # Градиенты по alpha
        d_alpha = self.V @ d_out # (3,)
        
        # Градиенты через Softmax
        # d_s_hat = alpha * (d_alpha - sum(alpha * d_alpha))
        sum_term = np.sum(self.alpha * d_alpha)
        d_s_hat = self.alpha * (d_alpha - sum_term) # (3,)
        
        # Градиенты через Scale
        d_s = d_s_hat / np.sqrt(d) # (3,)
        
        # Градиенты по Query
        dq = self.K.T @ d_s # (2,)
        
        # Градиенты по Keys
        dK = np.outer(d_s, self.q) # (3, 2)
        
        return dq, dK, dV

    def update(self, dq, dK, dV, eta):
        self.q -= eta * dq
        self.K -= eta * dK
        self.V -= eta * dV

# Демонстрация
if __name__ == '__main__':
    q = [1.0, 0.0]
    K = [[1.0, 1.0], [0.0, 2.0], [-1.0, 1.0]]
    V = [[1.0, 2.0], [3.0, 0.0], [0.0, 1.0]]
    t = np.array([1.0, 1.0])
    eta = 0.1

    net = Attention1D(q, K, V)
    out, s_hat, alpha = net.forward()
    
    print("Прямой проход:")
    print(f"  Scores (scaled): {np.round(s_hat, 4)}")
    print(f"  Weights (alpha): {np.round(alpha, 4)}")
    print(f"  Output:          {np.round(out, 4)}")
    
    dq, dK, dV = net.backward(out, t)
    print("\nОбратный проход:")
    print(f"  dq: {np.round(dq, 4)}")
    print(f"  dK: \n{np.round(dK, 4)}")
    print(f"  dV: \n{np.round(dV, 4)}")
    
    net.update(dq, dK, dV, eta)
    print("\nПосле шага:")
    print(f"  q: {np.round(net.q, 4)}")
    print(f"  K: \n{np.round(net.K, 4)}")
    print(f"  V: \n{np.round(net.V, 4)}")
```

## 1.9. Что мы получили

В этой главе мы разобрали «сердце» Трансформера:

1. **Query, Key, Value:** Механизм, который позволяет одному вектору (Query) вычислить важность (Weight) каждого элемента из базы (Key) и собрать нужную информацию (Value).
2. **Масштабирование:** Зачем нужно делить на \(\sqrt{d}\) (чтобы Softmax не выдавал жесткие нули и единицы).
3. **Softmax Backprop:** Мы вывели сложную производную Softmax в матричном виде. Это фундамент для обучения любых нейросетей с вниманием.
4. **Градиенты:** Мы увидели, как ошибка от выхода \(d\text{out}\) распределяется по значениям \(v\), затем по весам \(\alpha\), затем по ключам \(k\) и, наконец, по запросу \(q\).

**Что дальше?**
В этой главе \(q, k, v\) были просто заданными векторами. В следующей главе (Глава 2. Self-Attention) мы увидим, как эти векторы получаются из обычных входных слов с помощью матриц весов, и как вся последовательность слов обрабатывается одновременно. Это будет переход от «1D-свёртки» к «2D-свёртке» в мире внимания!


# Глава 2. Self-Attention на матрицах: как слова смотрят друг на друга

В предыдущей главе \(q, k, v\) были заданными векторами. Но откуда они берутся в реальной сети? В Трансформерах мы берем входную последовательность слов (например, эмбеддинги слов "Я", "люблю", "MLP") и с помощью **обучаемых матриц весов** \(W_Q, W_K, W_V\) превращаем их в матрицы Запросов (\(Q\)), Ключей (\(K\)) и Значений (\(V\)).

## 2.1. Зачем нужен этот пример

В Главе 1 у нас был один Query и три Key-Value. В реальности:
1.  У нас есть **последовательность** из \(L\) слов. Каждое слово представлено вектором длины \(d_{model}\).
2.  Каждое слово генерирует свой Query, свой Key и свой Value.
3.  Механизм внимания позволяет каждому слову «посмотреть» на все остальные слова (включая себя), чтобы понять контекст.

Главное отличие от Главы 1 — **построчный Softmax**. Каждое слово имеет свой собственный набор весов внимания (свою строку в матрице \(\alpha\)).

## 2.2. Архитектура и обозначения (Рис. 2.1)

| Символ | Смысл | Размерность |
|---|---|---|
| \(X\) | Входная последовательность (матрица эмбеддингов) | \(L \times d_{model}\) |
| \(W_Q, W_K, W_V\) | Обучаемые матрицы весов | \(d_{model} \times d_k\) |
| \(Q, K, V\) | Матрицы Запросов, Ключей, Значений | \(L \times d_k\) |
| \(S\) | Матрица сырых оценок (Scores): \(Q K^T\) | \(L \times L\) |
| \(S_{scaled}\) | Масштабированные оценки | \(L \times L\) |
| \(A\) | Матрица весов внимания (Attention weights) | \(L \times L\) |
| \(O\) | Выходная матрица | \(L \times d_v\) |
| \(T\) | Истинные ответы (Target) | \(L \times d_v\) |
| \(L_{loss}\) | Функция потерь (MSE) | Скаляр |

Для примера возьмем:
*   Последовательность из **2-х слов** (\(L=2\)).
*   Размерность эмбеддинга \(d_{model}=2\).
*   Размерность \(Q, K, V\): \(d_k = d_v = 2\).

Итого у нас **три матрицы весов** \(W_Q, W_K, W_V\) (каждая \(2 \times 2\)), то есть \(12\) обучаемых параметров.

```python
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def draw_self_attention():
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.set_xlim(0, 16); ax.set_ylim(0, 9)
    ax.axis('off')

    # Вход X
    ax.text(2, 8, 'Вход $X$ (2x2)', ha='center', fontsize=12, style='italic')
    for i in range(2):
        for j in range(2):
            ax.add_patch(patches.Rectangle((1 + j*0.8, 6.5 - i*0.8), 0.7, 0.7, facecolor='#AEDFF7', edgecolor='black'))
            ax.text(1.35 + j*0.8, 6.85 - i*0.8, f'\(x_{{{i+1}{j+1}}}\)', ha='center', va='center', fontsize=10)

    # Матрицы весов
    ax.text(5.5, 8, 'Веса \(W_Q, W_K, W_V\)', ha='center', fontsize=12, style='italic')
    for k, name in enumerate(['Q', 'K', 'V']):
        for i in range(2):
            for j in range(2):
                ax.add_patch(patches.Rectangle((4.5 + j*0.6 + k*1.5, 6.5 - i*0.6), 0.5, 0.5, facecolor='#FFE29A', edgecolor='black'))
                ax.text(4.75 + j*0.6 + k*1.5, 6.75 - i*0.6, f'\(w^{name}_{{{i+1}{j+1}}}\)', ha='center', va='center', fontsize=8)

    # Q, K, V
    ax.text(11, 8, 'Матрицы \(Q, K, V\) (2x2)', ha='center', fontsize=12, style='italic')
    for k, name in enumerate(['Q', 'K', 'V']):
        for i in range(2):
            for j in range(2):
                ax.add_patch(patches.Rectangle((9.5 + j*0.6 + k*1.5, 6.5 - i*0.6), 0.5, 0.5, facecolor='#B8E0B8', edgecolor='black'))
                ax.text(9.75 + j*0.6 + k*1.5, 6.75 - i*0.6, f'${name}_{{{i+1}{j+1}}}$', ha='center', va='center', fontsize=8)

    # Стрелки
    ax.annotate('', xy=(4.2, 6.5), xytext=(3.0, 6.5), arrowprops=dict(arrowstyle='->', color='black'))
    ax.annotate('', xy=(9.2, 6.5), xytext=(8.0, 6.5), arrowprops=dict(arrowstyle='->', color='black'))

    # Scores и Attention
    ax.text(5.5, 4, 'Матрица \(S = Q K^T\) (2x2)', ha='center', fontsize=12, style='italic')
    for i in range(2):
        for j in range(2):
            ax.add_patch(patches.Rectangle((4.5 + j*0.8, 2.5 - i*0.8), 0.7, 0.7, facecolor='#F5B7B1', edgecolor='black'))
            ax.text(4.85 + j*0.8, 2.85 - i*0.8, f'\(s_{{{i+1}{j+1}}}\)', ha='center', va='center', fontsize=10)

    ax.text(11, 4, 'Веса \(A = \\text{softmax}(S/\\sqrt{d_k})\)', ha='center', fontsize=12, style='italic')
    for i in range(2):
        for j in range(2):
            ax.add_patch(patches.Rectangle((9.5 + j*0.8, 2.5 - i*0.8), 0.7, 0.7, facecolor='#E8E8E8', edgecolor='black'))
            ax.text(9.85 + j*0.8, 2.85 - i*0.8, f'$\\alpha_{{{i+1}{j+1}}}$', ha='center', va='center', fontsize=10)

    ax.annotate('', xy=(4.2, 3.0), xytext=(6.0, 6.0), arrowprops=dict(arrowstyle='->', color='black'))
    ax.annotate('', xy=(9.2, 3.0), xytext=(7.5, 3.0), arrowprops=dict(arrowstyle='->', color='black'))

    ax.text(8.0, 0.5, 'Рис. 2.1. Self-Attention: каждое слово проецируется в Q, K, V, затем вычисляются веса внимания.',
            ha='center', fontsize=13, fontweight='bold')

    plt.tight_layout(); plt.show()

draw_self_attention()
```

## 2.3. Прямой проход (Forward Pass)

### Шаг 1. Вычисление \(Q, K, V\)
Мы умножаем входную матрицу \(X\) на обучаемые веса. Это обычные полносвязные слои (без смещения):
$$
Q = X W_Q, \quad K = X W_K, \quad V = X W_V.
$$
*(Размерность: $(L \times d_{model}) \times (d_{model} \times d_k) = L \times d_k$).*

### Шаг 2. Оценки внимания (Scores)
Сравниваем каждый Query с каждым Key. Поскольку у нас матрицы, мы используем матричное умножение \(Q\) на транспонированную \(K\):
$$
S = Q K^T.
$$
\(S_{i,j}\) — это насколько \(i\)-е слово обращает внимание на \(j\)-е слово.

### Шаг 3. Масштабирование и Softmax
Делим на \(\sqrt{d_k}\) (где \(d_k\) — размерность векторов \(Q, K\)):
$$
S_{scaled} = \frac{S}{\sqrt{d_k}}.
$$
Теперь применяем Softmax **построчно** (для каждой строки \(i\) сумма весов должна быть равна 1):
$$
A_{i,j} = \frac{e^{S_{scaled, i,j}}}{\sum_{k=1}^{L} e^{S_{scaled, i,k}}}.
$$
Строка \(i\) матрицы \(A\) показывает, на какие слова смотрит \(i\)-е слово.

### Шаг 4. Выход
Умножаем веса внимания на матрицу значений:
$$
O = A V.
$$
Строка \(i\) матрицы \(O\) — это «контекстно-обогащенный» вектор для \(i\)-го слова.

### Шаг 5. Функция потерь
Сравниваем матрицу \(O\) с целевой матрицей \(T\) (размерности \(L \times d_v\)):
$$
L_{loss} = \frac{1}{2} \sum_{i=1}^{L} \sum_{j=1}^{d_v} (O_{i,j} - T_{i,j})^2.
$$

## 2.4. Обратный проход (Backpropagation)

Нам нужно найти производные по \(W_Q, W_K, W_V\). Мы будем двигаться с конца и применять матричное цепное правило.

### Шаг 1. Градиент по выходу (\(dO\))
$$
dO = O - T. \quad (\text{размерность } L \times d_v)
$$

### Шаг 2. Градиенты по \(V\) и \(A\)
Из \(O = A V\):
$$
dV = A^T dO, \quad (\text{размерность } L \times d_v)
$$
$$
dA = dO V^T. \quad (\text{размерность } L \times L)
$$

### Шаг 3. Обратный проход через Softmax
Это самый тонкий момент. Мы применяем формулу Softmax **независимо для каждой строки**:
$$
dS_{scaled, i,j} = A_{i,j} \left( dA_{i,j} - \sum_{k=1}^{L} A_{i,k} dA_{i,k} \right).
$$
*(Смысл: мы вычитаем взвешенную сумму градиентов по строке).*

### Шаг 4. Обратный проход через масштабирование
$$
dS = \frac{dS_{scaled}}{\sqrt{d_k}}.
$$

### Шаг 5. Градиенты по \(Q\) и \(K\)
Из \(S = Q K^T\):
$$
dQ = dS K, \quad (\text{размерность } L \times d_k)
$$
$$
dK = dS^T Q. \quad (\text{размерность } L \times d_k)
$$
*(Вывод: \(dS_{i,j} = \sum_m dQ_{i,m} K_{j,m}\), откуда \(dQ = dS K\).)*

### Шаг 6. Градиенты по весам \(W_Q, W_K, W_V\)
Из \(Q = X W_Q\), \(K = X W_K\), \(V = X W_V\):
$$
dW_Q = X^T dQ, \quad dW_K = X^T dK, \quad dW_V = X^T dV.
$$
*(Размерность: \((d_{model} \times L) \times (L \times d_k) = d_{model} \times d_k\).)*

### Шаг 7. Обновление параметров
$$
W_Q \leftarrow W_Q - \eta dW_Q, \quad W_K \leftarrow W_K - \eta dW_K, \quad W_V \leftarrow W_V - \eta dW_V.
$$

## 2.5. Числовой пример

Для простоты возьмем \(L=2\), \(d_{model}=2\), \(d_k=2\).
*   Вход \(X = \begin{pmatrix} 1 & 0 \\ 0 & 2 \end{pmatrix}\).
*   Веса \(W_Q = W_K = W_V = \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix}\) (единичные матрицы, чтобы не усложнять).
*   Тогда \(Q = X, K = X, V = X\).
*   Цель \(T = \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix}\).
*   Шаг \(\eta = 0.1\).

### Прямой проход
1.  \(S = Q K^T = \begin{pmatrix} 1 & 0 \\ 0 & 2 \end{pmatrix} \begin{pmatrix} 1 & 0 \\ 0 & 2 \end{pmatrix} = \begin{pmatrix} 1 & 0 \\ 0 & 4 \end{pmatrix}\).
2.  \(S_{scaled} = S / \sqrt{2} \approx \begin{pmatrix} 0.707 & 0 \\ 0 & 2.828 \end{pmatrix}\).
3.  Softmax по строкам:
    *   Строка 1: \(\text{softmax}(0.707, 0) = (0.6698, 0.3302)\).
    *   Строка 2: \(\text{softmax}(0, 2.828) = (0.0552, 0.9448)\).
    \(A = \begin{pmatrix} 0.6698 & 0.3302 \\ 0.0552 & 0.9448 \end{pmatrix}\).
4.  \(O = A V = \begin{pmatrix} 0.6698 & 0.6604 \\ 0.0552 & 1.8896 \end{pmatrix}\).
5.  Потеря \(L_{loss} \approx 0.6695\).

### Обратный проход
1.  \(dO = O - T = \begin{pmatrix} -0.3302 & 0.6604 \\ 0.0552 & 0.8896 \end{pmatrix}\).
2.  \(dV = A^T dO = \begin{pmatrix} 0.6698 & 0.0552 \\ 0.3302 & 0.9448 \end{pmatrix} \begin{pmatrix} -0.3302 & 0.6604 \\ 0.0552 & 0.8896 \end{pmatrix} = \begin{pmatrix} -0.2182 & 0.4914 \\ -0.0382 & 1.0585 \end{pmatrix}\).
3.  \(dA = dO V^T = \begin{pmatrix} -0.3302 & 0.6604 \\ 0.0552 & 0.8896 \end{pmatrix} \begin{pmatrix} 1 & 0 \\ 0 & 2 \end{pmatrix} = \begin{pmatrix} -0.3302 & 1.3208 \\ 0.0552 & 1.7792 \end{pmatrix}\).
4.  Softmax backward по строкам:
    *   Строка 1: \(\sum A_{1k} dA_{1k} = 0.6698(-0.3302) + 0.3302(1.3208) \approx 0.215\).
        \(dS_{scaled, 1} = (0.6698(-0.3302 - 0.215), 0.3302(1.3208 - 0.215)) \approx (-0.365, 0.365)\).
    *   Строка 2: \(\sum A_{2k} dA_{2k} = 0.0552(0.0552) + 0.9448(1.7792) \approx 1.684\).
        \(dS_{scaled, 2} = (0.0552(0.0552 - 1.684), 0.9448(1.7792 - 1.684)) \approx (-0.090, 0.090)\).
    \(dS_{scaled} = \begin{pmatrix} -0.365 & 0.365 \\ -0.090 & 0.090 \end{pmatrix}\).
5.  \(dS = dS_{scaled} / \sqrt{2} = \begin{pmatrix} -0.258 & 0.258 \\ -0.064 & 0.064 \end{pmatrix}\).
6.  \(dQ = dS K = \begin{pmatrix} -0.258 & 0.258 \\ -0.064 & 0.064 \end{pmatrix} \begin{pmatrix} 1 & 0 \\ 0 & 2 \end{pmatrix} = \begin{pmatrix} -0.258 & 0.516 \\ -0.064 & 0.128 \end{pmatrix}\).
7.  \(dK = dS^T Q = \begin{pmatrix} -0.258 & -0.064 \\ 0.258 & 0.064 \end{pmatrix} \begin{pmatrix} 1 & 0 \\ 0 & 2 \end{pmatrix} = \begin{pmatrix} -0.258 & -0.128 \\ 0.258 & 0.128 \end{pmatrix}\).
8.  \(dW_Q = X^T dQ = \begin{pmatrix} 1 & 0 \\ 0 & 2 \end{pmatrix} \begin{pmatrix} -0.258 & 0.516 \\ -0.064 & 0.128 \end{pmatrix} = \begin{pmatrix} -0.258 & 0.516 \\ -0.128 & 0.256 \end{pmatrix}\).
9.  \(dW_K = X^T dK = \begin{pmatrix} 1 & 0 \\ 0 & 2 \end{pmatrix} \begin{pmatrix} -0.258 & -0.128 \\ 0.258 & 0.128 \end{pmatrix} = \begin{pmatrix} -0.258 & -0.128 \\ 0.516 & 0.256 \end{pmatrix}\).
10. \(dW_V = X^T dV = \begin{pmatrix} 1 & 0 \\ 0 & 2 \end{pmatrix} \begin{pmatrix} -0.2182 & 0.4914 \\ -0.0382 & 1.0585 \end{pmatrix} = \begin{pmatrix} -0.2182 & 0.4914 \\ -0.0764 & 2.1170 \end{pmatrix}\).

**Обновление:**
$$
W_Q \leftarrow \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix} - 0.1 \begin{pmatrix} -0.258 & 0.516 \\ -0.128 & 0.256 \end{pmatrix} = \begin{pmatrix} 1.0258 & -0.0516 \\ 0.0128 & 0.9744 \end{pmatrix}.
$$

## 2.6. Полная программа

```python
import numpy as np

def softmax_rows(S):
    """Применяет softmax к каждой строке матрицы S."""
    e_S = np.exp(S - np.max(S, axis=1, keepdims=True))
    return e_S / np.sum(e_S, axis=1, keepdims=True)

class SelfAttention:
    def __init__(self, X, W_Q, W_K, W_V):
        self.X = np.array(X, dtype=float)
        self.W_Q = np.array(W_Q, dtype=float)
        self.W_K = np.array(W_K, dtype=float)
        self.W_V = np.array(W_V, dtype=float)

    def forward(self):
        self.Q = self.X @ self.W_Q
        self.K = self.X @ self.W_K
        self.V = self.X @ self.W_V
        
        d_k = self.Q.shape[1]
        self.S = self.Q @ self.K.T
        self.S_scaled = self.S / np.sqrt(d_k)
        self.A = softmax_rows(self.S_scaled)
        self.O = self.A @ self.V
        return self.O

    def backward(self, T):
        d_k = self.Q.shape[1]
        dO = self.O - T
        
        dV = self.A.T @ dO
        dA = dO @ self.V.T
        
        # Softmax backward (построчно)
        # sum(A * dA) по каждой строке
        sum_term = np.sum(self.A * dA, axis=1, keepdims=True)
        dS_scaled = self.A * (dA - sum_term)
        
        dS = dS_scaled / np.sqrt(d_k)
        
        dQ = dS @ self.K
        dK = dS.T @ self.Q
        
        dW_Q = self.X.T @ dQ
        dW_K = self.X.T @ dK
        dW_V = self.X.T @ dV
        
        return dW_Q, dW_K, dW_V

    def update(self, dW_Q, dW_K, dW_V, eta):
        self.W_Q -= eta * dW_Q
        self.W_K -= eta * dW_K
        self.W_V -= eta * dW_V

# Демонстрация
if __name__ == '__main__':
    X = [[1, 0], [0, 2]]
    W_Q = [[1, 0], [0, 1]]
    W_K = [[1, 0], [0, 1]]
    W_V = [[1, 0], [0, 1]]
    T = [[1, 0], [0, 1]]
    eta = 0.1

    net = SelfAttention(X, W_Q, W_K, W_V)
    O = net.forward()
    
    print("=== Прямой проход ===")
    print(f"Q:\n{np.round(net.Q, 4)}")
    print(f"K:\n{np.round(net.K, 4)}")
    print(f"V:\n{np.round(net.V, 4)}")
    print(f"S:\n{np.round(net.S, 4)}")
    print(f"S_scaled:\n{np.round(net.S_scaled, 4)}")
    print(f"A (веса внимания):\n{np.round(net.A, 4)}")
    print(f"O (выход):\n{np.round(O, 4)}")
    
    dW_Q, dW_K, dW_V = net.backward(T)
    
    print("\n=== Обратный проход ===")
    print(f"dW_Q:\n{np.round(dW_Q, 4)}")
    print(f"dW_K:\n{np.round(dW_K, 4)}")
    print(f"dW_V:\n{np.round(dW_V, 4)}")
    
    net.update(dW_Q, dW_K, dW_V, eta)
    print("\n=== После шага ===")
    print(f"W_Q:\n{np.round(net.W_Q, 4)}")
    print(f"W_K:\n{np.round(net.W_K, 4)}")
    print(f"W_V:\n{np.round(net.W_V, 4)}")
```

## 2.7. Что мы получили

В этой главе мы перешли от одного вектора к **целой последовательности**:

1. **Матрицы весов:** Мы научились получать \(Q, K, V\) из входных данных с помощью обучаемых матриц \(W_Q, W_K, W_V\).
2. **Матричное внимание:** Вместо скалярного произведения мы используем \(Q K^T\). Это позволяет каждому слову «посмотреть» на все остальные слова одновременно (глобальный контекст).
3. **Построчный Softmax:** Мы применили Softmax к каждой строке матрицы \(S\). В обратном проходе это потребовало вычитания средневзвешенной суммы **для каждой строки отдельно**.
4. **Градиенты:** Мы вывели, как ошибка от выхода \(O\) возвращается к весам \(W_Q, W_K, W_V\) через цепочку: \(dO \to dA \to dS_{scaled} \to dS \to dQ, dK \to dW\).

**Что дальше?**
Теперь у нас есть полноценный Self-Attention. Но в реальных Трансформерах используют **Multi-Head Attention** (Многоголовое внимание). Это когда мы делаем несколько таких операций параллельно с разными матрицами весов (как несколько фильтров в CNN). Мы разберем его в следующей главе. Готовы?



# Глава 3. Multi-Head Attention: несколько взглядов на данные

В Главе 2 каждое слово генерировало один Query, один Key и один Value. Теперь мы заставим сеть генерировать **несколько** наборов \(Q, K, V\) для каждого слова.

Каждая «голова» (head) будет иметь свои собственные обучаемые матрицы весов \(W_Q^{(i)}, W_K^{(i)}, W_V^{(i)}\). Это позволит каждой голове фокусироваться на разных аспектах входных данных.

## 3.1. Зачем нужен этот пример

Многоголовое внимание (Multi-Head Attention) — это стандарт де-факто в архитектуре Трансформеров (BERT, GPT, LLaMA).
Главные отличия от Главы 2:
- **Разделение размерности:** Вместо одной большой операции мы делаем \(h\) маленьких. Размерность каждой головы \(d_k = d_{model} / h\).
- **Независимые веса:** Каждая голова имеет свои \(W_Q^{(i)}, W_K^{(i)}, W_V^{(i)}\).
- **Конкатенация:** Результаты всех голов склеиваются в один вектор.
- **Выходная проекция:** Склеенный вектор умножается на финальную матрицу \(W_O\), чтобы вернуть исходную размерность.

## 3.2. Архитектура и обозначения (Рис. 3.1)

| Символ | Смысл | Размерность |
|---|---|---|
| \(X\) | Входная матрица | \(L \times d_{model}\) |
| \(h\) | Количество голов (heads) | Скаляр |
| \(d_k, d_v\) | Размерность внутри одной головы | \(d_{model} / h\) |
| \(W_Q^{(i)}, W_K^{(i)}, W_V^{(i)}\) | Веса \(i\)-й головы | \(d_{model} \times d_k\) |
| \(Q^{(i)}, K^{(i)}, V^{(i)}\) | Матрицы \(i\)-й головы | \(L \times d_k\) |
| \(\text{head}_i\) | Выход \(i\)-й головы | \(L \times d_v\) |
| \(\text{Concat}\) | Склеенная матрица всех голов | \(L \times (h \cdot d_v)\) |
| \(W_O\) | Выходная проекция | \((h \cdot d_v) \times d_{model}\) |
| \(O\) | Финальный выход | \(L \times d_{model}\) |

Для примера возьмем:
*   Последовательность \(L=2\).
*   \(d_{model}=2\).
*   Количество голов \(h=2\). Тогда \(d_k = d_v = 1\).
*   \(W_Q^{(1)}, W_K^{(1)}, W_V^{(1)}\) — матрицы \(2 \times 1\) (векторы-столбцы).
*   \(W_O\) — матрица \(2 \times 2\).

```python
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def draw_multi_head_attention():
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.set_xlim(0, 18); ax.set_ylim(0, 10)
    ax.axis('off')

    # Вход X
    ax.text(2, 9, 'Вход $X$', ha='center', fontsize=12, style='italic')
    ax.add_patch(patches.Rectangle((1, 7), 2, 1.5, facecolor='#AEDFF7', edgecolor='black'))
    
    # Разветвление на 2 головы
    ax.text(6, 9, 'Голова 1', ha='center', fontsize=12, style='italic', color='#E74C3C')
    ax.text(12, 9, 'Голова 2', ha='center', fontsize=12, style='italic', color='#2980B9')

    for i, (x_off, color) in enumerate([(4.5, '#E74C3C'), (10.5, '#2980B9')]):
        for j, name in enumerate(['Q', 'K', 'V']):
            ax.add_patch(patches.Rectangle((x_off + j*1.2, 7), 0.8, 1.5, facecolor='#FFE29A', edgecolor='black'))
            ax.text(x_off + 0.4 + j*1.2, 7.75, f'{name}⁽{i+1}⁾', ha='center', va='center', fontsize=10)
        
        # Стрелка к Attention
        ax.annotate('', xy=(x_off + 1.8, 5.5), xytext=(x_off + 1.8, 6.8), arrowprops=dict(arrowstyle='->', color=color, lw=2))
        ax.add_patch(patches.Rectangle((x_off + 0.6, 4), 2.4, 1.2, facecolor='#E8E8E8', edgecolor=color, lw=2))
        ax.text(x_off + 1.8, 4.6, 'Attention', ha='center', va='center', fontsize=10, color=color)

    # Конкатенация
    ax.annotate('', xy=(8, 2.5), xytext=(6.3, 3.8), arrowprops=dict(arrowstyle='->', color='#E74C3C', lw=2))
    ax.annotate('', xy=(8, 2.5), xytext=(12.3, 3.8), arrowprops=dict(arrowstyle='->', color='#2980B9', lw=2))
    
    ax.add_patch(patches.Rectangle((7, 1.5), 2, 1, facecolor='#B8E0B8', edgecolor='black', lw=2))
    ax.text(8, 2.0, 'Concat', ha='center', va='center', fontsize=12)

    # Выходная проекция
    ax.annotate('', xy=(11, 2.0), xytext=(9.2, 2.0), arrowprops=dict(arrowstyle='->', color='black', lw=2))
    ax.add_patch(patches.Rectangle((11, 1.2), 1.2, 1.6, facecolor='#F5B7B1', edgecolor='black', lw=2))
    ax.text(11.6, 2.0, '\(W_O\)', ha='center', va='center', fontsize=12)

    ax.annotate('', xy=(14, 2.0), xytext=(12.4, 2.0), arrowprops=dict(arrowstyle='->', color='black', lw=2))
    ax.add_patch(patches.Rectangle((14, 1.2), 1.5, 1.6, facecolor='#AEDFF7', edgecolor='black', lw=2))
    ax.text(14.75, 2.0, '\(O\)', ha='center', va='center', fontsize=14)

    ax.text(8.0, 0.0, 'Рис. 3.1. Multi-Head Attention: несколько независимых голов, затем конкатенация и проекция.',
            ha='center', fontsize=13, fontweight='bold')

    plt.tight_layout(); plt.show()

draw_multi_head_attention()
```

## 3.3. Прямой проход (Forward Pass)

### Шаг 1. Проекции для каждой головы
Для каждой головы \(i = 1, \dots, h\) вычисляем свои \(Q, K, V\):
$$
Q^{(i)} = X W_Q^{(i)}, \quad K^{(i)} = X W_K^{(i)}, \quad V^{(i)} = X W_V^{(i)}.
$$
*(Размерность: \((L \times d_{model}) \times (d_{model} \times d_k) = L \times d_k\)).*

### Шаг 2. Внимание внутри головы
Для каждой головы считаем стандартный Self-Attention (как в Главе 2):
$$
S^{(i)} = Q^{(i)} (K^{(i)})^T,
$$
$$
A^{(i)} = \text{softmax}\left( \frac{S^{(i)}}{\sqrt{d_k}} \right),
$$
$$
\text{head}_i = A^{(i)} V^{(i)}.
$$

### Шаг 3. Конкатенация
Склеиваем результаты всех голов по последней размерности (по признакам):
$$
\text{Concat} = [\text{head}_1, \text{head}_2, \dots, \text{head}_h].
$$
*(Размерность: \(L \times (h \cdot d_v)\)).*

### Шаг 4. Выходная проекция
Умножаем склеенную матрицу на \(W_O\):
$$
O = \text{Concat} \cdot W_O.
$$
*(Размерность: \((L \times h d_v) \times (h d_v \times d_{model}) = L \times d_{model}\)).*

## 3.4. Обратный проход (Backpropagation)

Нам нужно найти градиенты по всем \(W_Q^{(i)}, W_K^{(i)}, W_V^{(i)}\) и \(W_O\).

### Шаг 1. Градиент по выходу (\(dO\))
$$
dO = O - T. \quad (L \times d_{model})
$$

### Шаг 2. Градиенты для \(W_O\) и \(\text{Concat}\)
Из \(O = \text{Concat} \cdot W_O\):
$$
dW_O = \text{Concat}^T dO, \quad (h d_v \times d_{model})
$$
$$
d\text{Concat} = dO W_O^T. \quad (L \times h d_v)
$$

### Шаг 3. Разделение (Split) градиента по головам
Мы «разрезаем» \(d\text{Concat}\) на \(h\) частей, соответствующих каждой голове:
$$
d\text{head}_i = \text{Split}_i(d\text{Concat}). \quad (L \times d_v)
$$

### Шаг 4. Обратный проход внутри каждой головы
Для каждой головы \(i\) мы применяем формулы из Главы 2. Сначала находим \(dA^{(i)}\) и \(dV^{(i)}\):
$$
dV^{(i)} = (A^{(i)})^T d\text{head}_i,
$$
$$
dA^{(i)} = d\text{head}_i (V^{(i)})^T.
$$
Затем проходим через Softmax:
$$
dS_{scaled}^{(i)} = A^{(i)} \odot \left( dA^{(i)} - \sum_{k} A^{(i)}_{*,k} dA^{(i)}_{*,k} \right),
$$
где сумма берется по строкам.
Масштабирование и \(Q, K\):
$$
dS^{(i)} = \frac{dS_{scaled}^{(i)}}{\sqrt{d_k}},
$$
$$
dQ^{(i)} = dS^{(i)} K^{(i)},
$$
$$
dK^{(i)} = (dS^{(i)})^T Q^{(i)}.
$$

### Шаг 5. Градиенты для весов голов
$$
dW_Q^{(i)} = X^T dQ^{(i)}, \quad dW_K^{(i)} = X^T dK^{(i)}, \quad dW_V^{(i)} = X^T dV^{(i)}.
$$

## 3.5. Числовой пример

Для простоты возьмем:
*   \(X = \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix}\) (два слова, размерность 2).
*   \(h = 2\). Значит \(d_k = 1\).
*   \(W_Q^{(1)} = W_K^{(1)} = W_V^{(1)} = \begin{pmatrix} 1 \\ 0 \end{pmatrix}\) (первая голова смотрит только на первую размерность).
*   \(W_Q^{(2)} = W_K^{(2)} = W_V^{(2)} = \begin{pmatrix} 0 \\ 1 \end{pmatrix}\) (вторая голова смотрит только на вторую размерность).
*   \(W_O = \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix}\) (просто пропускает сигнал).
*   Цель \(T = \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix}\).

### Прямой проход
**Голова 1:**
\(Q^{(1)} = X W_Q^{(1)} = \begin{pmatrix} 1 \\ 0 \end{pmatrix}\). \(K^{(1)} = \begin{pmatrix} 1 \\ 0 \end{pmatrix}\), \(V^{(1)} = \begin{pmatrix} 1 \\ 0 \end{pmatrix}\).
\(S^{(1)} = Q^{(1)} (K^{(1)})^T = \begin{pmatrix} 1 \\ 0 \end{pmatrix} \begin{pmatrix} 1 & 0 \end{pmatrix} = \begin{pmatrix} 1 & 0 \\ 0 & 0 \end{pmatrix}\).
\(A^{(1)} = \text{softmax}(\begin{pmatrix} 1 & 0 \\ 0 & 0 \end{pmatrix}) = \begin{pmatrix} 0.731 & 0.269 \\ 0.5 & 0.5 \end{pmatrix}\).
\(\text{head}_1 = A^{(1)} V^{(1)} = \begin{pmatrix} 0.731 \\ 0.5 \end{pmatrix}\).

**Голова 2:**
\(Q^{(2)} = \begin{pmatrix} 0 \\ 1 \end{pmatrix}\). \(K^{(2)} = \begin{pmatrix} 0 \\ 1 \end{pmatrix}\), \(V^{(2)} = \begin{pmatrix} 0 \\ 1 \end{pmatrix}\).
\(S^{(2)} = \begin{pmatrix} 0 & 0 \\ 0 & 1 \end{pmatrix}\).
\(A^{(2)} = \begin{pmatrix} 0.5 & 0.5 \\ 0.269 & 0.731 \end{pmatrix}\).
\(\text{head}_2 = \begin{pmatrix} 0.5 \\ 0.731 \end{pmatrix}\).

**Конкатенация:**
\(\text{Concat} = \begin{pmatrix} 0.731 & 0.5 \\ 0.5 & 0.731 \end{pmatrix}\).

**Выход:**
\(O = \text{Concat} \cdot W_O = \begin{pmatrix} 0.731 & 0.5 \\ 0.5 & 0.731 \end{pmatrix}\).

Потеря: \(L_{loss} \approx \frac{1}{2}(0.269^2 + 0.5^2 + 0.5^2 + 0.269^2) \approx 0.322\).

### Обратный проход
\(dO = O - T = \begin{pmatrix} -0.269 & 0.5 \\ 0.5 & -0.269 \end{pmatrix}\).

\(dW_O = \text{Concat}^T dO = \begin{pmatrix} 0.731 & 0.5 \\ 0.5 & 0.731 \end{pmatrix} \begin{pmatrix} -0.269 & 0.5 \\ 0.5 & -0.269 \end{pmatrix} = \begin{pmatrix} 0.053 & 0.231 \\ 0.231 & 0.053 \end{pmatrix}\).

\(d\text{Concat} = dO W_O^T = \begin{pmatrix} -0.269 & 0.5 \\ 0.5 & -0.269 \end{pmatrix}\).

**Разделение (Split):**
\(d\text{head}_1 = \begin{pmatrix} -0.269 \\ 0.5 \end{pmatrix}\), \(d\text{head}_2 = \begin{pmatrix} 0.5 \\ -0.269 \end{pmatrix}\).

**Обратный проход Головы 1:**
\(dV^{(1)} = (A^{(1)})^T d\text{head}_1 = \begin{pmatrix} 0.731 & 0.5 \\ 0.269 & 0.5 \end{pmatrix} \begin{pmatrix} -0.269 \\ 0.5 \end{pmatrix} = \begin{pmatrix} 0.053 \\ 0.177 \end{pmatrix}\).
\(dA^{(1)} = d\text{head}_1 (V^{(1)})^T = \begin{pmatrix} -0.269 \\ 0.5 \end{pmatrix} \begin{pmatrix} 1 & 0 \end{pmatrix} = \begin{pmatrix} -0.269 & 0 \\ 0.5 & 0 \end{pmatrix}\).
Softmax backward по строкам:
Строка 1: \(A_{1} \odot (dA_1 - \text{sum}) = \begin{pmatrix} 0.731 \\ 0.269 \end{pmatrix} \odot (\begin{pmatrix} -0.269 \\ 0 \end{pmatrix} - \begin{pmatrix} -0.197 \\ 0 \end{pmatrix}) = \begin{pmatrix} -0.053 \\ 0 \end{pmatrix}\).
Строка 2: аналогично \(\begin{pmatrix} 0.125 \\ 0.125 \end{pmatrix}\).
\(dS_{scaled}^{(1)} = \begin{pmatrix} -0.053 & 0 \\ 0.125 & 0.125 \end{pmatrix}\).
\(dS^{(1)} = dS_{scaled}^{(1)} / 1 = \begin{pmatrix} -0.053 & 0 \\ 0.125 & 0.125 \end{pmatrix}\).
\(dQ^{(1)} = dS^{(1)} K^{(1)} = \begin{pmatrix} -0.053 & 0 \\ 0.125 & 0.125 \end{pmatrix} \begin{pmatrix} 1 \\ 0 \end{pmatrix} = \begin{pmatrix} -0.053 \\ 0.125 \end{pmatrix}\).
\(dK^{(1)} = (dS^{(1)})^T Q^{(1)} = \begin{pmatrix} -0.053 & 0.125 \\ 0 & 0.125 \end{pmatrix} \begin{pmatrix} 1 \\ 0 \end{pmatrix} = \begin{pmatrix} -0.053 \\ 0 \end{pmatrix}\).
\(dW_Q^{(1)} = X^T dQ^{(1)} = \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix} \begin{pmatrix} -0.053 \\ 0.125 \end{pmatrix} = \begin{pmatrix} -0.053 \\ 0.125 \end{pmatrix}\).
И так далее для остальных голов и весов.

## 3.6. Полная программа

```python
import numpy as np

def softmax(x):
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e_x / np.sum(e_x, axis=-1, keepdims=True)

class MultiHeadAttention:
    def __init__(self, X, num_heads, W_Q_list, W_K_list, W_V_list, W_O):
        self.X = np.array(X, dtype=float)
        self.h = num_heads
        self.W_Q_list = [np.array(W, dtype=float) for W in W_Q_list]
        self.W_K_list = [np.array(W, dtype=float) for W in W_K_list]
        self.W_V_list = [np.array(W, dtype=float) for W in W_V_list]
        self.W_O = np.array(W_O, dtype=float)

    def forward(self):
        self.heads = []
        self.Q_list, self.K_list, self.V_list, self.A_list = [], [], [], []
        
        for i in range(self.h):
            Q = self.X @ self.W_Q_list[i]
            K = self.X @ self.W_K_list[i]
            V = self.X @ self.W_V_list[i]
            
            d_k = Q.shape[1]
            S = Q @ K.T
            S_scaled = S / np.sqrt(d_k)
            A = softmax(S_scaled)
            head = A @ V
            
            self.Q_list.append(Q)
            self.K_list.append(K)
            self.V_list.append(V)
            self.A_list.append(A)
            self.heads.append(head)
            
        self.Concat = np.concatenate(self.heads, axis=1)
        self.O = self.Concat @ self.W_O
        return self.O

    def backward(self, T):
        dO = self.O - T
        
        dW_O = self.Concat.T @ dO
        dConcat = dO @ self.W_O.T
        
        # Разделение (Split)
        d_heads = np.split(dConcat, self.h, axis=1)
        
        self.dW_Q_list, self.dW_K_list, self.dW_V_list = [], [], []
        
        for i in range(self.h):
            d_head = d_heads[i]
            
            dV = self.A_list[i].T @ d_head
            dA = d_head @ self.V_list[i].T
            
            # Softmax backward
            sum_term = np.sum(self.A_list[i] * dA, axis=1, keepdims=True)
            dS_scaled = self.A_list[i] * (dA - sum_term)
            
            d_k = self.Q_list[i].shape[1]
            dS = dS_scaled / np.sqrt(d_k)
            
            dQ = dS @ self.K_list[i]
            dK = dS.T @ self.Q_list[i]
            
            self.dW_Q_list.append(self.X.T @ dQ)
            self.dW_K_list.append(self.X.T @ dK)
            self.dW_V_list.append(self.X.T @ dV)
            
        return self.dW_Q_list, self.dW_K_list, self.dW_V_list, dW_O

    def update(self, eta):
        for i in range(self.h):
            self.W_Q_list[i] -= eta * self.dW_Q_list[i]
            self.W_K_list[i] -= eta * self.dW_K_list[i]
            self.W_V_list[i] -= eta * self.dW_V_list[i]
        self.W_O -= eta * self.dW_O

# Демонстрация (используем пример из §3.5)
if __name__ == '__main__':
    X = [[1, 0], [0, 1]]
    # Для примера W_Q^(1) и W_Q^(2) заданы как векторы-столбцы (2x1)
    W_Q1 = [[1], [0]]
    W_K1 = [[1], [0]]
    W_V1 = [[1], [0]]
    
    W_Q2 = [[0], [1]]
    W_K2 = [[0], [1]]
    W_V2 = [[0], [1]]
    
    W_O = [[1, 0], [0, 1]]
    T = [[1, 0], [0, 1]]
    eta = 0.1

    net = MultiHeadAttention(X, 2, [W_Q1, W_Q2], [W_K1, W_K2], [W_V1, W_V2], W_O)
    O = net.forward()
    
    print("=== Прямой проход ===")
    print(f"Concat:\n{np.round(net.Concat, 4)}")
    print(f"O (выход):\n{np.round(O, 4)}")
    
    dW_Q_list, dW_K_list, dW_V_list, dW_O = net.backward(T)
    
    print("\n=== Обратный проход ===")
    print(f"dW_Q1:\n{np.round(dW_Q_list[0], 4)}")
    print(f"dW_Q2:\n{np.round(dW_Q_list[1], 4)}")
    print(f"dW_O:\n{np.round(dW_O, 4)}")
```

## 3.7. Что мы получили

В этой главе мы разобрали архитектуру **Multi-Head Attention**:

1. **Разделение на головы:** Входные данные проецируются \(h\) раз с разными весами, позволяя сети улавливать разные паттерны.
2. **Параллельное внимание:** Каждая голова вычисляет свой Self-Attention независимо.
3. **Конкатенация:** Результаты склеиваются в один вектор.
4. **Выходная проекция:** Матрица \(W_O\) смешивает информацию от разных голов перед подачей на следующий слой.
5. **Обратный проход:** Самое интересное здесь — как градиент \(d\text{Concat}\) аккуратно «разрезается» на части (\(Split\)), чтобы каждая голова получила свою долю ошибки.

Теперь у нас есть «двигатель» Трансформера. Осталось добавить **Positional Encoding** (чтобы сеть знала порядок слов) и **Layer Normalization** (чтобы сеть могла быть глубокой). Об этом — в следующих главах.


# Глава 4. Positional Encoding: как научить сеть понимать порядок слов

## 4.1. Зачем нужен этот пример

В отличие от рекуррентных сетей (RNN), которые читают текст слева направо (и уже знают порядок), или сверточных сетей (CNN), которые смотрят на локальные окна, механизм Self-Attention **абсолютно не зависит от порядка**.

Если мы просто поменяем местами векторы слов в матрице \(X\), результат Self-Attention математически не изменится (изменится только порядок строк). Чтобы решить эту проблему, мы должны **внедрить информацию о позиции** прямо в векторы слов.

Самый простой способ — сложить вектор слова с вектором его позиции:
$$
X_{in} = X + PE
$$
где \(X\) — исходные эмбеддинги слов, а \(PE\) — матрица позиционного кодирования (Positional Encoding). Мы будем использовать синусоиды и косинусы разных частот, чтобы сеть могла легко выучить относительные расстояния между словами.

## 4.2. Архитектура и обозначения (Рис. 4.1)

| Символ | Смысл | Размерность |
|---|---|---|
| \(X\) | Входные эмбеддинги слов | \(L \times d_{model}\) |
| \(pos\) | Позиция слова в предложении (от 0 до \(L-1\)) | Скаляр |
| \(i\) | Индекс размерности (от 0 до \(d_{model}/2 - 1\)) | Скаляр |
| \(PE_{(pos, 2i)}\) | Значение синуса для \(pos\) и четного индекса \(2i\) | Скаляр |
| \(PE_{(pos, 2i+1)}\) | Значение косинуса для \(pos\) и нечетного индекса \(2i+1\) | Скаляр |
| \(PE\) | Матрица позиционного кодирования | \(L \times d_{model}\) |
| \(X_{in}\) | Итоговый вход, который пойдет в Self-Attention | \(L \times d_{model}\) |

Для примера возьмем предложение из **двух слов** (\(L=2\)) и размерность модели \(d_{model}=4\).

```python
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

def draw_positional_encoding():
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.set_xlim(0, 16); ax.set_ylim(0, 7)
    ax.axis('off')

    # Вход X
    ax.text(2.0, 6.0, 'Эмбеддинги слов $X$', ha='center', fontsize=12, style='italic')
    for i in range(2):
        for j in range(4):
            ax.add_patch(patches.Rectangle((1 + j*0.8, 4.5 - i*0.8), 0.7, 0.7, facecolor='#AEDFF7', edgecolor='black'))
            ax.text(1.35 + j*0.8, 4.85 - i*0.8, f'\(x_{{{i+1}{j+1}}}\)', ha='center', va='center', fontsize=10)

    # Плюс
    ax.text(6.0, 4.5, '+', ha='center', va='center', fontsize=30)

    # PE
    ax.text(9.0, 6.0, 'Позиционное кодирование $PE$', ha='center', fontsize=12, style='italic')
    for i in range(2):
        for j in range(4):
            ax.add_patch(patches.Rectangle((8 + j*0.8, 4.5 - i*0.8), 0.7, 0.7, facecolor='#FFE29A', edgecolor='black'))
            ax.text(8.35 + j*0.8, 4.85 - i*0.8, f'\(pe_{{{i+1}{j+1}}}\)', ha='center', va='center', fontsize=10)

    # Равно
    ax.text(12.0, 4.5, '=', ha='center', va='center', fontsize=30)

    # Выход X_in
    ax.text(15.0, 6.0, 'Итоговый вход $X_{in}$', ha='center', fontsize=12, style='italic')
    for i in range(2):
        for j in range(4):
            ax.add_patch(patches.Rectangle((14 + j*0.8, 4.5 - i*0.8), 0.7, 0.7, facecolor='#B8E0B8', edgecolor='black'))
            ax.text(14.35 + j*0.8, 4.85 - i*0.8, f'\(a_{{{i+1}{j+1}}}\)', ha='center', va='center', fontsize=10)

    ax.text(8.0, 1.5, 'Рис. 4.1. Positional Encoding: мы просто складываем эмбеддинги слов с векторами их позиций.',
            ha='center', fontsize=13, fontweight='bold')

    plt.tight_layout(); plt.show()

draw_positional_encoding()
```

## 4.3. Прямой проход (Forward Pass)

### Шаг 1. Формула позиционного кодирования
Для каждой позиции \(pos\) и каждого измерения \(i\) (где \(i\) идет от \(0\) до \(d_{model}/2 - 1\)), мы вычисляем:
$$
PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)
$$
$$
PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)
$$

*Почему именно так?* Частота \(\frac{1}{10000^{2i/d_{model}}}\) уменьшается с ростом \(i\). Это создает «двоичный счетчик»: для малых \(i\) синусоида колеблется быстро, для больших \(i\) — медленно. Это позволяет сети легко выучить относительные расстояния, так как \(PE_{pos+k}\) является линейной функцией от \(PE_{pos}\).

### Шаг 2. Сложение
Мы просто прибавляем матрицу \(PE\) (размер \(L \times d_{model}\)) к матрице \(X\):
$$
X_{in} = X + PE.
$$
Это поэлементное сложение. Например, \(a_{1,1} = x_{1,1} + pe_{1,1}\).

## 4.4. Функция потерь

Сама по себе матрица \(PE\) не имеет функции потерь. Она используется внутри более крупной сети (например, в Трансформере, который предсказывает следующее слово). Поэтому мы предполагаем, что от следующего слоя (например, от Multi-Head Attention) к нам пришел градиент ошибки по \(X_{in}\):
$$
dX_{in} = \frac{\partial L}{\partial X_{in}}.
$$

## 4.5. Обратный проход (Backpropagation)

Обратный проход через Positional Encoding — это самый простой шаг во всем Трансформере, потому что это **операция сложения**.

### Шаг 1. Градиент по \(X\)
Поскольку \(X_{in} = X + PE\), производная от \(X_{in}\) по \(X\) равна \(1\). По правилу сложения градиентов (как в ResNet):
$$
dX = dX_{in} \cdot 1 = dX_{in}.
$$
То есть, градиент от следующего слоя просто передается на вход без изменений.

### Шаг 2. Градиент по \(PE\)
Аналогично, производная от \(X_{in}\) по \(PE\) равна \(1\):
$$
dPE = dX_{in} \cdot 1 = dX_{in}.
$$
*(Примечание: Если \(PE\) — это фиксированная матрица синусов (как в оригинальном Трансформере), то мы не обновляем её, и на этом обратный проход заканчивается. Если \(PE\) — это обучаемые параметры, как в BERT, то мы обновляем \(PE \leftarrow PE - \eta \cdot dPE\)).*

**Именно из-за этой простоты мы и используем сложение, а не конкатенацию. Сложение позволяет градиенту течь напрямую.**

## 4.6. Числовой пример

Для простоты возьмем \(L=2\) (два слова), \(d_{model}=4\).
Слова: \(X = \begin{pmatrix} 1 & 0 & 1 & 0 \\ 0 & 1 & 0 & 1 \end{pmatrix}\).

### Прямой проход
Рассчитаем \(PE\) для \(pos=0\) (первое слово) и \(pos=1\) (второе слово).
\(d_{model} = 4\), значит \(2i\) меняется как \(0, 2\) (для синусов) и \(2i+1\) как \(1, 3\) (для косинусов).

**Слово 1 (pos=0):**
*   \(i=0\): \(\sin(0 / 10000^{0}) = \sin(0) = 0\). \(\cos(0) = 1\).
*   \(i=1\): \(\sin(0 / 10000^{2/4}) = \sin(0) = 0\). \(\cos(0) = 1\).
*   Вектор \(PE_1 = [0, 1, 0, 1]\).

**Слово 2 (pos=1):**
*   \(i=0\): \(\sin(1 / 10000^{0}) = \sin(1) \approx 0.8415\). \(\cos(1) \approx 0.5403\).
*   \(i=1\): \(\sin(1 / 10000^{2/4}) = \sin(1/100) = \sin(0.01) \approx 0.0100\). \(\cos(0.01) \approx 0.99995\).
*   Вектор \(PE_2 = [0.8415, 0.5403, 0.0100, 0.99995]\).

**Матрица \(PE\):**
$$
PE = \begin{pmatrix} 0 & 1 & 0 & 1 \\ 0.8415 & 0.5403 & 0.0100 & 0.99995 \end{pmatrix}.
$$

**Сложение:**
$$
X_{in} = X + PE = \begin{pmatrix} 1+0 & 0+1 & 1+0 & 0+1 \\ 0+0.8415 & 1+0.5403 & 0+0.0100 & 1+0.99995 \end{pmatrix} = \begin{pmatrix} 1 & 1 & 1 & 1 \\ 0.8415 & 1.5403 & 0.0100 & 1.99995 \end{pmatrix}.
$$

### Обратный проход
Пусть от механизма Self-Attention пришел градиент:
$$
dX_{in} = \begin{pmatrix} 0.1 & -0.1 & 0.2 & -0.2 \\ -0.1 & 0.1 & -0.2 & 0.2 \end{pmatrix}.
$$
Тогда по правилу сложения:
$$
dX = dX_{in}, \quad dPE = dX_{in}.
$$
*(Мы просто копируем матрицу \(dX_{in}\) в \(dX\) и \(dPE\)).*

## 4.7. Полная программа

```python
import numpy as np

class PositionalEncoding:
    def __init__(self, d_model, max_len=5000):
        self.d_model = d_model
        self.max_len = max_len
        self.PE = self._generate_pe()

    def _generate_pe(self):
        PE = np.zeros((self.max_len, self.d_model))
        for pos in range(self.max_len):
            for i in range(0, self.d_model, 2):
                # Формула синуса
                PE[pos, i] = np.sin(pos / (10000 ** (i / self.d_model)))
                # Формула косинуса
                if i + 1 < self.d_model:
                    PE[pos, i+1] = np.cos(pos / (10000 ** (i / self.d_model)))
        return PE

    def forward(self, X):
        # X имеет размерность (L, d_model)
        L = X.shape[0]
        self.X = X
        self.PE_slice = self.PE[:L, :]
        self.X_in = self.X + self.PE_slice
        return self.X_in

    def backward(self, dX_in):
        # Градиент проходит напрямую
        dX = dX_in
        dPE = dX_in
        return dX, dPE

# Демонстрация
if __name__ == '__main__':
    X = np.array([
        [1, 0, 1, 0],
        [0, 1, 0, 1]
    ], dtype=float)

    net = PositionalEncoding(d_model=4)
    X_in = net.forward(X)
    
    print("=== Прямой проход ===")
    print(f"PE (первые 2 строки):\n{np.round(net.PE_slice, 4)}")
    print(f"X_in (X + PE):\n{np.round(X_in, 4)}")
    
    # Искусственный градиент от следующего слоя
    dX_in = np.array([
        [0.1, -0.1, 0.2, -0.2],
        [-0.1, 0.1, -0.2, 0.2]
    ])
    
    dX, dPE = net.backward(dX_in)
    
    print("\n=== Обратный проход ===")
    print(f"dX (градиент для X):\n{np.round(dX, 4)}")
    print(f"dPE (градиент для PE):\n{np.round(dPE, 4)}")
```

## 4.8. Что мы получили

В этой главе мы решили проблему «мешка слов» в Трансформерах:

1. **Проблема порядка:** Self-Attention не видит разницы между перестановками слов.
2. **Решение:** Мы добавляем к эмбеддингам матрицу \(PE\) с синусами и косинусами разных частот.
3. **Магия синусов:** Благодаря тригонометрии, сеть может легко вычислить относительное расстояние между словами (например, что слово на позиции 5 находится на расстоянии 2 от слова на позиции 3).
4. **Обратный проход:** Поскольку это сложение, градиент просто передается назад без изменений. Это делает обучение очень быстрым.

Теперь у нас есть всё, чтобы собрать **Блок Энкодера (Encoder Block)**! В следующей главе мы соединим Multi-Head Attention, Positional Encoding и **Layer Normalization** (нормализацию слоя, которая стабилизирует обучение) в единый механизм. Готовы?



# Глава 5. Add & Norm: Остаточные связи и Слой нормализации

В этой главе мы разберем два важнейших компонента, которые позволяют строить 100-слойные Трансформеры (как GPT-4):

1. **Residual Connection:** Мы просто прибавляем вход слоя к его выходу (\(Y = X + Z\)). Это создает «шорткат» для градиента, позволяя ему течь напрямую, не затухая.
2. **Layer Normalization:** Мы нормализуем вектор каждого слова отдельно (вычитаем среднее, делим на стандартное отклонение), чтобы активации всегда были в стабильном диапазоне.

## 5.1. Зачем нужен этот пример

Если мы применим только Attention, распределение значений на выходе может сильно меняться от слоя к слою. Layer Normalization «причесывает» данные, а Residual Connection гарантирует, что даже если новый слой обучится плохо (будет выдавать нули), сеть сможет просто «перепрыгнуть» его и передать исходные данные дальше.

## 5.2. Архитектура и обозначения (Рис. 5.1)

| Символ | Смысл | Размерность |
|---|---|---|
| \(X\) | Входной тензор (от предыдущего слоя) | \(L \times d_{model}\) |
| \(Z\) | Выход подслоя (например, Multi-Head Attention) | \(L \times d_{model}\) |
| \(Y\) | Сумма \(X + Z\) | \(L \times d_{model}\) |
| \(\mu\) | Вектор средних значений для каждого слова | \(L \times 1\) |
| \(\sigma^2\) | Вектор дисперсий для каждого слова | \(L \times 1\) |
| \(\hat{Y}\) | Нормализованный тензор | \(L \times d_{model}\) |
| \(\gamma\) | Обучаемый параметр масштаба (Scale) | \(d_{model}\) |
| \(\beta\) | Обучаемый параметр сдвига (Shift) | \(d_{model}\) |
| \(Out\) | Итоговый выход (нормализованный) | \(L \times d_{model}\) |

Для примера возьмем **одно слово** (\(L=1\)) и размерность \(d_{model}=3\), чтобы числа были наглядными.

```python
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyArrowPatch

def draw_add_norm():
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.set_xlim(0, 14); ax.set_ylim(0, 8)
    ax.axis('off')

    # Вход X
    ax.add_patch(patches.Rectangle((1, 3), 1.5, 3, facecolor='#AEDFF7', edgecolor='black'))
    ax.text(1.75, 4.5, '\(X\)', ha='center', va='center', fontsize=16)
    
    # Выход подслоя Z
    ax.add_patch(patches.Rectangle((4, 3), 1.5, 3, facecolor='#FFE29A', edgecolor='black'))
    ax.text(4.75, 4.5, '\(Z\)', ha='center', va='center', fontsize=16)

    # Сложение
    ax.add_patch(patches.Circle((7.5, 4.5), 0.5, facecolor='#E8E8E8', edgecolor='black'))
    ax.text(7.5, 4.5, '+', ha='center', va='center', fontsize=20)

    # LayerNorm
    ax.add_patch(patches.Rectangle((9.5, 3), 1.5, 3, facecolor='#B8E0B8', edgecolor='black'))
    ax.text(10.25, 4.5, 'LayerNorm\n\((\gamma, \beta)\)', ha='center', va='center', fontsize=10)

    # Выход
    ax.add_patch(patches.Rectangle((12.5, 3.5), 1, 2, facecolor='#F5B7B1', edgecolor='black'))
    ax.text(13.0, 4.5, '\(Out\)', ha='center', va='center', fontsize=16)

    # Стрелки
    ax.annotate('', xy=(7.0, 4.5), xytext=(5.6, 4.5), arrowprops=dict(arrowstyle='->', color='black', lw=2))
    ax.annotate('', xy=(7.0, 5.2), xytext=(2.5, 5.2), arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=2, connectionstyle="arc3,rad=-0.3"))
    ax.text(4.5, 6.0, 'Residual Connection', fontsize=10, color='#2C3E50')
    
    ax.annotate('', xy=(9.4, 4.5), xytext=(8.1, 4.5), arrowprops=dict(arrowstyle='->', color='black', lw=2))
    ax.annotate('', xy=(12.4, 4.5), xytext=(11.1, 4.5), arrowprops=dict(arrowstyle='->', color='black', lw=2))

    ax.text(7.0, 0.5, 'Рис. 5.1. Блок Add & Norm: сложение входа с выходом подслоя и нормализация.',
            ha='center', fontsize=13, fontweight='bold')

    plt.tight_layout(); plt.show()

draw_add_norm()
```

## 5.3. Прямой проход (Forward Pass)

### Шаг 1. Residual Connection (Сложение)
Сначала мы складываем вход \(X\) и выход подслоя \(Z\) (например, Attention):
$$
Y_{i,j} = X_{i,j} + Z_{i,j}.
$$
*(Здесь \(i\) — индекс слова в последовательности, \(j\) — индекс признака).*

### Шаг 2. Вычисление среднего (Mean)
Для **каждого слова** \(i\) мы считаем среднее значение по всем его признакам \(j\):
$$
\mu_i = \frac{1}{d} \sum_{j=1}^{d} Y_{i,j}.
$$

### Шаг 3. Вычисление дисперсии (Variance)
Считаем дисперсию (разброс) для каждого слова \(i\):
$$
\sigma_i^2 = \frac{1}{d} \sum_{j=1}^{d} (Y_{i,j} - \mu_i)^2.
$$

### Шаг 4. Нормализация (Normalization)
Вычитаем среднее и делим на стандартное отклонение (с добавлением крошечного \(\epsilon\) для численной стабильности, чтобы не делить на ноль):
$$
\hat{Y}_{i,j} = \frac{Y_{i,j} - \mu_i}{\sqrt{\sigma_i^2 + \epsilon}}.
$$
Теперь каждое слово имеет среднее 0 и дисперсию 1.

### Шаг 5. Масштаб и Сдвиг (Scale & Shift)
Нормализация могла «убить» полезную информацию, поэтому мы даем сети возможность вернуть нужный масштаб. \(\gamma\) и \(\beta\) — это **обучаемые** векторы длины \(d_{model}\) (общие для всех слов):
$$
Out_{i,j} = \gamma_j \cdot \hat{Y}_{i,j} + \beta_j.
$$

## 5.4. Функция потерь

Допустим, от следующего слоя к нам пришел градиент ошибки по выходу:
$$
dOut_{i,j} = \frac{\partial L}{\partial Out_{i,j}}.
$$

## 5.5. Обратный проход (Backpropagation) — Полный вывод

Это самый сложный обратный проход, потому что нормализация зависит от **всех** элементов слова сразу.

### Шаг 1. Градиенты по \(\gamma\) и \(\beta\)
Для \(j\)-го признака (суммируем по всем словам \(i\)):
$$
\frac{\partial L}{\partial \gamma_j} = \sum_{i} dOut_{i,j} \cdot \hat{Y}_{i,j},
$$
$$
\frac{\partial L}{\partial \beta_j} = \sum_{i} dOut_{i,j}.
$$

### Шаг 2. Градиент по нормализованному \(\hat{Y}\)
$$
d\hat{Y}_{i,j} = dOut_{i,j} \cdot \gamma_j.
$$

### Шаг 3. Градиенты по дисперсии (\(\sigma^2\)) и среднему (\(\mu\))
Здесь начинается математика. Нам нужно применить цепное правило к формулам Шагов 2-4.
Сначала градиент по дисперсии (суммируем по признакам \(k\)):
$$
d\sigma_i^2 = \sum_{k=1}^{d} d\hat{Y}_{i,k} \cdot (Y_{i,k} - \mu_i) \cdot \left( -\frac{1}{2} (\sigma_i^2 + \epsilon)^{-3/2} \right).
$$
Затем градиент по среднему:
$$
d\mu_i = \sum_{k=1}^{d} d\hat{Y}_{i,k} \cdot \frac{-1}{\sqrt{\sigma_i^2 + \epsilon}} + d\sigma_i^2 \cdot \frac{1}{d}\sum_{k=1}^{d} -2(Y_{i,k} - \mu_i).
$$
*(Заметим, что \(\sum (Y_{i,k} - \mu_i) = 0\), поэтому второе слагаемое равно нулю).*

### Шаг 4. Градиент по \(Y\) (Итоговая формула)
Собираем всё вместе, чтобы получить градиент по исходной сумме \(Y\):
$$
\frac{\partial L}{\partial Y_{i,j}} = d\hat{Y}_{i,j} \cdot \frac{1}{\sqrt{\sigma_i^2 + \epsilon}} + d\sigma_i^2 \cdot \frac{2(Y_{i,j} - \mu_i)}{d} + d\mu_i \cdot \frac{1}{d}.
$$
Эту громоздкую формулу можно свернуть в изящное выражение:
$$
dY_{i,j} = \frac{1}{\sqrt{\sigma_i^2 + \epsilon}} \left( d\hat{Y}_{i,j} - \frac{1}{d}\sum_{k} d\hat{Y}_{i,k} - \frac{\hat{Y}_{i,j}}{d}\sum_{k} d\hat{Y}_{i,k} \hat{Y}_{i,k} \right).
$$
*(Смысл: мы вычитаем средний градиент и градиент по дисперсии, чтобы не нарушить нормализацию).*

### Шаг 5. Градиенты по \(X\) и \(Z\)
Так как \(Y_{i,j} = X_{i,j} + Z_{i,j}\), производные равны 1:
$$
dX_{i,j} = dY_{i,j}, \quad dZ_{i,j} = dY_{i,j}.
$$
Градиент просто дублируется на оба входа!

## 5.6. Числовой пример

Возьмем **одно слово** (\(L=1\)) размерности \(d=3\).
Пусть \(X = (1, 2, 3)\), \(Z = (3, 2, 1)\).
Тогда \(Y = X + Z = (4, 4, 4)\).

*Внимание:* дисперсия \(Y\) равна \(0\). Чтобы не делить на ноль, пусть \(Y\) будет чуть другим.
Возьмем \(Y = (1, 2, 3)\). (Это значит \(X = (0, 1, 1)\), \(Z = (1, 1, 2)\)).
\(\mu = (1+2+3)/3 = 2\).
\(\sigma^2 = ((1-2)^2 + (2-2)^2 + (3-2)^2) / 3 = 2/3 \approx 0.667\).
\(\epsilon = 0\).

**Нормализация:**
$$
\hat{Y} = \frac{(1, 2, 3) - 2}{\sqrt{0.667}} = (-1, 0, 1) / 0.8165 \approx (-1.2247, 0, 1.2247).
$$
Пусть \(\gamma = (2, 2, 2)\), \(\beta = (1, 1, 1)\).
Тогда \(Out = 2 \cdot \hat{Y} + 1 = (-1.4494, 1, 3.4494)\).

### Обратный проход
Пусть градиент от следующего слоя: \(dOut = (1, 1, 0)\).

**Шаг 1.** \(d\gamma = dOut \cdot \hat{Y} = (1(-1.2247), 1(0), 0(1.2247)) = (-1.2247, 0, 0)\).
\(d\beta = (1, 1, 0)\).

**Шаг 2.** \(d\hat{Y} = dOut \cdot \gamma = (2, 2, 0)\).

**Шаг 3.** Считаем градиент по \(Y\):
Сумма \(1\): \(\sum d\hat{Y} = 4\).
Сумма \(2\): \(\sum d\hat{Y} \cdot \hat{Y} = 2(-1.2247) + 2(0) + 0 = -2.4494\).
Для \(j=1\):
$$
dY_1 = \frac{1}{0.8165} \left( 2 - \frac{4}{3} - \frac{-1.2247}{3}(-2.4494) \right) = 1.2247 \left( 2 - 1.333 - 1.0 \right) = -0.408.
$$
Для \(j=2\):
$$
dY_2 = 1.2247 \left( 2 - 1.333 - 0 \right) = 0.816.
$$
Для \(j=3\):
$$
dY_3 = 1.2247 \left( 0 - 1.333 - \frac{1.2247}{3}(-2.4494) \right) = 1.2247 \left( -1.333 + 1.0 \right) = -0.408.
$$

**Шаг 4.** \(dX = dY = (-0.408, 0.816, -0.408)\), \(dZ = dY\).

## 5.7. Полная программа

```python
import numpy as np

class AddNorm:
    def __init__(self, d_model, eps=1e-5):
        self.eps = eps
        self.gamma = np.ones(d_model)
        self.beta = np.zeros(d_model)

    def forward(self, X, Z):
        # 1. Residual
        self.X = X
        self.Z = Z
        self.Y = X + Z
        
        # 2. Mean & Variance (по последней размерности)
        self.mu = np.mean(self.Y, axis=-1, keepdims=True)
        self.var = np.var(self.Y, axis=-1, keepdims=True)
        
        # 3. Normalize
        self.std = np.sqrt(self.var + self.eps)
        self.Y_hat = (self.Y - self.mu) / self.std
        
        # 4. Scale & Shift
        self.out = self.gamma * self.Y_hat + self.beta
        return self.out

    def backward(self, d_out):
        # d_out имеет размерность (L, d_model)
        # 1. d_gamma, d_beta (суммируем по L)
        self.d_gamma = np.sum(d_out * self.Y_hat, axis=0)
        self.d_beta = np.sum(d_out, axis=0)
        
        # 2. d_Y_hat
        d_Y_hat = d_out * self.gamma
        
        # 3. Свернутая формула для dY
        d = self.Y.shape[-1]
        sum_dY_hat = np.sum(d_Y_hat, axis=-1, keepdims=True)
        sum_dY_hat_Y_hat = np.sum(d_Y_hat * self.Y_hat, axis=-1, keepdims=True)
        
        d_Y = (1 / self.std) * (d_Y_hat - sum_dY_hat / d - self.Y_hat * sum_dY_hat_Y_hat / d)
        
        # 4. dX, dZ
        d_X = d_Y
        d_Z = d_Y
        
        return d_X, d_Z

    def update(self, d_gamma, d_beta, eta):
        self.gamma -= eta * d_gamma
        self.beta -= eta * d_beta

# Демонстрация
if __name__ == '__main__':
    X = np.array([[0.0, 1.0, 1.0]]) # (1, 3)
    Z = np.array([[1.0, 1.0, 2.0]]) # (1, 3)
    
    net = AddNorm(d_model=3)
    out = net.forward(X, Z)
    
    print("=== Прямой проход ===")
    print(f"Y (X+Z):\n{np.round(net.Y, 4)}")
    print(f"Mean:\n{np.round(net.mu, 4)}")
    print(f"Std:\n{np.round(net.std, 4)}")
    print(f"Out:\n{np.round(out, 4)}")
    
    # Градиент от следующего слоя (например, от Dense)
    d_out = np.array([[1.0, 1.0, 0.0]])
    d_X, d_Z = net.backward(d_out)
    
    print("\n=== Обратный проход ===")
    print(f"d_gamma:\n{np.round(net.d_gamma, 4)}")
    print(f"d_beta:\n{np.round(net.d_beta, 4)}")
    print(f"dX:\n{np.round(d_X, 4)}")
    print(f"dZ:\n{np.round(d_Z, 4)}")
```

## 5.8. Что мы получили

В этой главе мы разобрали блок **Add & Norm**:

1. **Residual Connection:** Мы научились прибавлять вход к выходу. Это решает проблему затухания градиента в глубоких сетях (благодаря \(dX = dY\)).
2. **Layer Normalization:** Мы нормализуем признаки каждого слова отдельно, чтобы стабилизировать обучение. В отличие от BatchNorm (который смотрит на весь батч), LayerNorm не зависит от размера батча.
3. **Обратный проход:** Мы вывели сложную формулу градиента через нормализацию. Она позволяет ошибке корректно распределяться между признаками одного слова.

Теперь у нас есть **все компоненты** для сборки полноценного Энкодера Трансформера. В следующей главе (Глава 6) мы объединим Multi-Head Attention, Positional Encoding и Add & Norm в единый **Encoder Block** и напишем полный обратный проход для всей этой конструкции. Это будет «венец» нашего курса по Энкодерам!



# Глава 6. Блок Энкодера (Encoder Block): собираем всё вместе

В предыдущих главах мы разобрали:
1. **Multi-Head Attention** (Глава 3) — как слова общаются друг с другом.
2. **Positional Encoding** (Глава 4) — как сеть узнает порядок слов.
3. **Add & Norm** (Глава 5) — как стабилизировать обучение.

Теперь мы соберем их в архитектуру, которая называется **Блок Энкодера**.

## 6.1. Зачем нужен этот пример

Сам по себе механизм Attention — это просто взвешенное суммирование. Чтобы сеть могла изучать сложные нелинейные зависимости (например, синтаксис и семантику языка), после Attention нужен обычный **полносвязный слой (Feed-Forward Network, FFN)**.

Блок Энкодера состоит из двух подслоев:
1. **Multi-Head Attention + Add & Norm**
2. **Feed-Forward Network (FFN) + Add & Norm**

Именно такой блок позволяет извлекать из текста признаки, которые потом пойдут в выходной слой (или в Декодер).

## 6.2. Архитектура и обозначения (Рис. 6.1)

Для примера возьмем одно слово (\(L=1\)) и размерность \(d_{model}=2\).
Внутри FFN мы сделаем скрытый слой размером \(d_{ff}=2\).

| Символ | Смысл | Размерность |
|---|---|---|
| \(X\) | Входной тензор (от предыдущего слоя) | \(L \times d_{model}\) |
| \(Z_1\) | Выход Multi-Head Attention | \(L \times d_{model}\) |
| \(X_1\) | Выход первого Add & Norm | \(L \times d_{model}\) |
| \(H\) | Скрытый слой FFN (после ReLU) | \(L \times d_{ff}\) |
| \(Z_2\) | Выход FFN (второй линейный слой) | \(L \times d_{model}\) |
| \(X_2\) | Выход второго Add & Norm (финал блока) | \(L \times d_{model}\) |
| \(W_1, b_1\) | Веса и смещение первой линейной части FFN | \(d_{model} \times d_{ff}\) |
| \(W_2, b_2\) | Веса и смещение второй линейной части FFN | \(d_{ff} \times d_{model}\) |

```python
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyArrowPatch

def draw_encoder_block():
    fig, ax = plt.subplots(figsize=(12, 9))
    ax.set_xlim(0, 14); ax.set_ylim(0, 11)
    ax.axis('off')

    # Вход
    ax.add_patch(patches.Rectangle((1, 5), 1.5, 2, facecolor='#AEDFF7', edgecolor='black', lw=2))
    ax.text(1.75, 6, '\(X\)', ha='center', va='center', fontsize=16)

    # Multi-Head Attention
    ax.add_patch(patches.Rectangle((4, 5), 2, 2, facecolor='#FFE29A', edgecolor='black', lw=2))
    ax.text(5.0, 6, 'Multi-Head\nAttention', ha='center', va='center', fontsize=12)
    
    # Add & Norm 1
    ax.add_patch(patches.Rectangle((7, 5), 1.5, 2, facecolor='#B8E0B8', edgecolor='black', lw=2))
    ax.text(7.75, 6, 'Add &\nNorm', ha='center', va='center', fontsize=12)

    # FFN
    ax.add_patch(patches.Rectangle((10, 5), 2, 2, facecolor='#FFE29A', edgecolor='black', lw=2))
    ax.text(11.0, 6, 'Feed\nForward', ha='center', va='center', fontsize=12)

    # Add & Norm 2
    ax.add_patch(patches.Rectangle((10, 2), 2, 1.5, facecolor='#B8E0B8', edgecolor='black', lw=2))
    ax.text(11.0, 2.75, 'Add &\nNorm', ha='center', va='center', fontsize=12)

    # Выход
    ax.add_patch(patches.Rectangle((7, 2), 1.5, 1.5, facecolor='#F5B7B1', edgecolor='black', lw=2))
    ax.text(7.75, 2.75, '\(X_2\)\n(выход)', ha='center', va='center', fontsize=12)

    # Стрелки
    ax.annotate('', xy=(4.0, 6), xytext=(2.6, 6), arrowprops=dict(arrowstyle='->', color='black', lw=2))
    ax.annotate('', xy=(7.0, 6), xytext=(6.1, 6), arrowprops=dict(arrowstyle='->', color='black', lw=2))
    ax.annotate('', xy=(10.0, 6), xytext=(8.6, 6), arrowprops=dict(arrowstyle='->', color='black', lw=2))
    
    # Residual 1 (вход в AddNorm 1)
    ax.annotate('', xy=(7.75, 7.1), xytext=(1.75, 7.1), arrowprops=dict(arrowstyle='->', color='#2980B9', lw=2, connectionstyle="arc3,rad=-0.2"))
    ax.text(4.75, 7.6, 'Residual', fontsize=10, color='#2980B9')

    # FFN вниз
    ax.annotate('', xy=(11.0, 5), xytext=(11.0, 4.5), arrowprops=dict(arrowstyle='->', color='black', lw=2))
    ax.annotate('', xy=(12.0, 3.5), xytext=(11.0, 4.5), arrowprops=dict(arrowstyle='->', color='black', lw=2))
    
    # Residual 2 (вход в AddNorm 2)
    ax.annotate('', xy=(12.1, 3.5), xytext=(8.6, 6), arrowprops=dict(arrowstyle='->', color='#2980B9', lw=2, connectionstyle="arc3,rad=0.3"))

    # Выход из AddNorm 2
    ax.annotate('', xy=(8.6, 2.75), xytext=(10.0, 2.75), arrowprops=dict(arrowstyle='->', color='black', lw=2))

    ax.text(7.0, 0.5, 'Рис. 6.1. Блок Энкодера: Attention + FFN с остаточными связями.',
            ha='center', fontsize=13, fontweight='bold')

    plt.tight_layout(); plt.show()

draw_encoder_block()
```

## 6.3. Прямой проход (Forward Pass)

### Шаг 1. Multi-Head Attention
Вход \(X\) (размер \(1 \times 2\)) подается в Multi-Head Attention (с матрицами \(W_Q, W_K, W_V\)). Получаем выход \(Z_1\) (размер \(1 \times 2\)).
$$
Z_1 = \text{Attention}(X).
$$

### Шаг 2. Первый Add & Norm
Складываем вход \(X\) и выход \(Z_1\):
$$
Y_1 = X + Z_1.
$$
Применяем LayerNorm к \(Y_1\):
$$
\mu_1 = \frac{1}{2} \sum_{j=1}^{2} Y_{1,j}, \quad \sigma_1^2 = \frac{1}{2} \sum_{j=1}^{2} (Y_{1,j} - \mu_1)^2,
$$
$$
X_1 = \text{LayerNorm}(Y_1) = \gamma_1 \odot \frac{Y_1 - \mu_1}{\sqrt{\sigma_1^2 + \epsilon}} + \beta_1.
$$
*(Для простоты можно считать, что \(\gamma_1 = 1, \beta_1 = 0\)).*

### Шаг 3. Feed-Forward Network (FFN)
Это обычный двухслойный перцептрон (MLP), который применяется к каждому слову **независимо**.
1. Линейный слой + ReLU:
$$
H = \text{ReLU}(X_1 W_1 + b_1).
$$
*(Размерность: \((1 \times 2) \times (2 \times 2) \to 1 \times 2\).)*
2. Второй линейный слой:
$$
Z_2 = H W_2 + b_2.
$$
*(Размерность: \((1 \times 2) \times (2 \times 2) \to 1 \times 2\).)*

### Шаг 4. Второй Add & Norm
Складываем выход FFN с выходом первого Add & Norm:
$$
Y_2 = X_1 + Z_2.
$$
Применяем LayerNorm:
$$
\mu_2 = \frac{1}{2} \sum_{j=1}^{2} Y_{2,j}, \quad \sigma_2^2 = \frac{1}{2} \sum_{j=1}^{2} (Y_{2,j} - \mu_2)^2,
$$
$$
X_2 = \gamma_2 \odot \frac{Y_2 - \mu_2}{\sqrt{\sigma_2^2 + \epsilon}} + \beta_2.
$$

Выход \(X_2\) — это результат работы всего Блока Энкодера.

## 6.4. Обратный проход (Backpropagation)

Пусть от следующего слоя к нам пришел градиент \(dX_2\) (размер \(1 \times 2\)). Нам нужно найти градиенты для \(W_Q, W_K, W_V, W_1, W_2, b_1, b_2\).

### Шаг 1. Обратный проход через второй Add & Norm
Из \(X_2\) мы получаем два градиента: для \(X_1\) и \(Z_2\).
Используем формулу из Главы 5 (производная LayerNorm):
$$
dY_2 = \text{LayerNormBackward}(dX_2).
$$
Поскольку \(Y_2 = X_1 + Z_2\):
$$
dX_1^{(1)} = dY_2, \quad dZ_2 = dY_2.
$$
*(Верхний индекс $(1)$ означает, что это первая часть градиента для \(X_1\)).*

### Шаг 2. Обратный проход через FFN
Зная \(dZ_2\), находим градиенты для \(W_2\) и \(b_2\):
$$
dW_2 = H^T dZ_2, \quad db_2 = dZ_2.
$$
Затем находим градиент по скрытому слою \(H\):
$$
dH = dZ_2 W_2^T.
$$
Проходим через ReLU (обнуляем градиенты там, где \(H \le 0\)):
$$
dX_1^{(2)} = (dH \odot \text{ReLU}'(X_1 W_1 + b_1)) W_1^T.
$$
Находим градиенты для \(W_1\) и \(b_1\):
$$
dW_1 = X_1^T (dH \odot \text{ReLU}'(X_1 W_1 + b_1)), \quad db_1 = dH \odot \text{ReLU}'(X_1 W_1 + b_1).
$$

### Шаг 3. Суммируем градиенты для \(X_1\)
В \(X_1\) приходят два потока градиентов: от Residual Connection (Шаг 1) и от FFN (Шаг 2).
$$
dX_1 = dX_1^{(1)} + dX_1^{(2)}.
$$

### Шаг 4. Обратный проход через первый Add & Norm
Применяем LayerNormBackward к \(dX_1\), получаем \(dY_1\).
Из \(Y_1 = X + Z_1\):
$$
dX^{(1)} = dY_1, \quad dZ_1 = dY_1.
$$

### Шаг 5. Обратный проход через Multi-Head Attention
Зная \(dZ_1\), мы применяем формулы из Главы 3 для нахождения \(dW_Q, dW_K, dW_V\) и \(dX^{(2)}\) (градиент от Attention).

### Шаг 6. Суммируем градиенты для \(X\)
$$
dX = dX^{(1)} + dX^{(2)}.
$$

Этот градиент \(dX\) передается на предыдущий слой сети.

## 6.5. Числовой пример

Пусть:
*   Вход \(X = (1, 2)\). (Это одно слово).
*   Attention сгенерировал \(Z_1 = (3, 1)\).
*   \(\gamma_1 = \gamma_2 = (1, 1)\), \(\beta_1 = \beta_2 = (0, 0)\).
*   FFN: \(W_1 = \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix}\), \(b_1 = (0, 0)\).
*   \(W_2 = \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix}\), \(b_2 = (0, 0)\).

### Прямой проход
**Add & Norm 1:**
\(Y_1 = X + Z_1 = (1, 2) + (3, 1) = (4, 3)\).
\(\mu_1 = 3.5\), \(\sigma_1^2 = 0.25\).
\(\hat{Y}_1 = \frac{(4, 3) - 3.5}{\sqrt{0.25}} = \frac{(0.5, -0.5)}{0.5} = (1, -1)\).
\(X_1 = (1, -1)\).

**FFN:**
\(H = \text{ReLU}(X_1 W_1 + b_1) = \text{ReLU}(1, -1) = (1, 0)\).
\(Z_2 = H W_2 + b_2 = (1, 0)\).

**Add & Norm 2:**
\(Y_2 = X_1 + Z_2 = (1, -1) + (1, 0) = (2, -1)\).
\(\mu_2 = 0.5\), \(\sigma_2^2 = 2.25\).
\(\hat{Y}_2 = \frac{(2, -1) - 0.5}{\sqrt{2.25}} = \frac{(1.5, -1.5)}{1.5} = (1, -1)\).
\(X_2 = (1, -1)\).

### Обратный проход
Пусть пришел градиент \(dX_2 = (1, 1)\).

**Шаг 1.** Находим \(dY_2\) через LayerNormBackward.
\(d = 2\), \(std = 1.5\).
\(d\hat{Y}_2 = dX_2 \cdot \gamma = (1, 1)\).
\(\sum d\hat{Y}_2 = 2\). \(\sum d\hat{Y}_2 \cdot \hat{Y}_2 = 1(1) + 1(-1) = 0\).
\(dY_2 = \frac{1}{1.5} \left( (1, 1) - \frac{2}{2} - \frac{(1, -1)}{2}(0) \right) = \frac{1}{1.5} (0, 0) = (0, 0)\).
*(Из-за симметрии градиент занулился).*
Тогда \(dX_1^{(1)} = (0, 0)\), \(dZ_2 = (0, 0)\).

**Шаг 2.** Обратный проход через FFN.
\(dW_2 = H^T dZ_2 = \begin{pmatrix} 1 \\ 0 \end{pmatrix} (0, 0) = \begin{pmatrix} 0 & 0 \\ 0 & 0 \end{pmatrix}\).
\(db_2 = (0, 0)\).
\(dH = dZ_2 W_2^T = (0, 0)\).
\(dX_1^{(2)} = (0, 0)\).

**Шаг 3.** \(dX_1 = dX_1^{(1)} + dX_1^{(2)} = (0, 0)\).

**Шаг 4.** Обратный проход через Add & Norm 1.
\(dY_1 = (0, 0)\). \(dX^{(1)} = (0, 0)\), \(dZ_1 = (0, 0)\).

**Шаг 5.** Градиенты Attention равны нулю.

*(В реальной сети градиенты не будут нулевыми, так как начальные веса случайны и не идеальны. Здесь мы взяли слишком симметричный пример).*

## 6.6. Полная программа

```python
import numpy as np

# (Предполагаем, что классы MultiHeadAttention, AddNorm и FFN уже написаны ранее)
# Для полноты картины соберем FFN здесь.

class FeedForward:
    def __init__(self, d_model, d_ff):
        self.W1 = np.random.randn(d_model, d_ff) * 0.1
        self.b1 = np.zeros(d_ff)
        self.W2 = np.random.randn(d_ff, d_model) * 0.1
        self.b2 = np.zeros(d_model)

    def forward(self, X):
        self.X = X
        self.H_pre = X @ self.W1 + self.b1
        self.H = np.maximum(0, self.H_pre) # ReLU
        self.Z = self.H @ self.W2 + self.b2
        return self.Z

    def backward(self, dZ):
        # dZ имеет размерность (L, d_model)
        self.dW2 = self.H.T @ dZ
        self.db2 = np.sum(dZ, axis=0)
        
        dH = dZ @ self.W2.T
        dH_pre = dH * (self.H_pre > 0) # Производная ReLU
        
        self.dW1 = self.X.T @ dH_pre
        self.db1 = np.sum(dH_pre, axis=0)
        
        dX = dH_pre @ self.W1.T
        return dX

    def update(self, eta):
        self.W1 -= eta * self.dW1
        self.b1 -= eta * self.db1
        self.W2 -= eta * self.dW2
        self.b2 -= eta * self.db2

class EncoderBlock:
    def __init__(self, d_model, d_ff):
        # В реальности здесь был бы MultiHeadAttention
        # Мы создаем заглушку, чтобы показать архитектуру
        self.attn = None
        self.norm1 = AddNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm2 = AddNorm(d_model)

    def forward(self, X):
        # 1. Attention + Add & Norm
        # (Представим, что Attention вернул Z1)
        Z1 = np.random.randn(*X.shape) # Заглушка
        X1 = self.norm1.forward(X, Z1)
        
        # 2. FFN + Add & Norm
        Z2 = self.ffn.forward(X1)
        X2 = self.norm2.forward(X1, Z2)
        return X2

    def backward(self, dX2):
        # 1. Назад через Add & Norm 2
        dX1_from_norm2, dZ2 = self.norm2.backward(dX2)
        
        # 2. Назад через FFN
        dX1_from_ffn = self.ffn.backward(dZ2)
        
        # 3. Суммируем градиенты для X1
        dX1 = dX1_from_norm2 + dX1_from_ffn
        
        # 4. Назад через Add & Norm 1
        dX_from_norm1, dZ1 = self.norm1.backward(dX1)
        
        # 5. Назад через Attention (заглушка)
        dX_from_attn = np.zeros_like(dX_from_norm1)
        
        # 6. Суммируем градиенты для X
        dX = dX_from_norm1 + dX_from_attn
        return dX

    def update(self, eta):
        self.norm1.update(self.norm1.d_gamma, self.norm1.d_beta, eta)
        self.ffn.update(eta)
        self.norm2.update(self.norm2.d_gamma, self.norm2.d_beta, eta)

if __name__ == '__main__':
    np.random.seed(42)
    X = np.array([[1.0, 2.0]])
    d_model, d_ff = 2, 4
    
    block = EncoderBlock(d_model, d_ff)
    out = block.forward(X)
    
    print(f"Вход X:\n{X}")
    print(f"Выход блока X2:\n{np.round(out, 4)}")
    
    dX2 = np.array([[1.0, 1.0]])
    dX = block.backward(dX2)
    print(f"\nГрадиент на входе dX:\n{np.round(dX, 4)}")
```

## 6.7. Что мы получили

В этой главе мы собрали **Блок Энкодера** — главный строительный элемент Трансформера:

1. **Структура:** Внимание (для связи слов) + FFN (для нелинейной обработки).
2. **Стабильность:** Два блока Add & Norm (остаточные связи и нормализация).
3. **Прямой проход:** Данные последовательно проходят через Attention, FFN и нормализации.
4. **Обратный проход:** Самое сложное здесь — правильно **сложить градиенты** на входах \(X_1\) и \(X\), так как градиент приходит как от Residual Connection, так и от подслоя (Attention или FFN).

Теперь у нас есть готовый «кирпичик». Если мы поставим 6 таких блоков подряд (как в оригинальном Трансформере), мы получим **полноценный Энкодер**.

В следующей (финальной) главе мы разберем **Декодер** (как сеть генерирует текст) и узнаем, что такое **Masked Attention** (как запретить сети подглядывать в будущее).



# Глава 7. Декодер: Masked Self-Attention и Cross-Attention

В предыдущих главах мы разобрали Self-Attention (когда слово смотрит на все слова в предложении). Но в Декодере это правило нарушается.

## 7.1. Зачем нужен этот пример

Представьте, что вы переводите фразу «Я люблю MLP» на английский. Энкодер прочитал всё предложение целиком и выдал свои скрытые состояния (контекст). Декодер начинает генерировать перевод: «I», затем «love», затем «MLP».
Когда Декодер генерирует слово «love», он **не должен** видеть слово «MLP» (оно же еще не сгенерировано!). Но он **должен** видеть слова «I» (которые уже сгенерировал) и **должен** смотреть на Энкодер (чтобы знать, что речь идет о любви к MLP).

Именно для этого нужны два новых механизма:
- **Masked Self-Attention:** Маскирует (зануляет) веса для будущих слов.
- **Cross-Attention:** Берет Query из Декодера, а Key и Value — из Энкодера.

## 7.2. Архитектура и обозначения (Рис. 7.1)

| Символ | Смысл | Размерность |
|---|---|---|
| \(X_{dec}\) | Вход Декодера (уже сгенерированные слова) | \(L_{dec} \times d_{model}\) |
| \(X_{enc}\) | Выход Энкодера (контекст исходного текста) | \(L_{enc} \times d_{model}\) |
| \(Q_{dec}, K_{dec}, V_{dec}\) | Проекции Декодера для Masked Attention | \(L_{dec} \times d_k\) |
| \(M\) | Маска (верхний треугольник = \(-\infty\), остальное = 0) | \(L_{dec} \times L_{dec}\) |
| \(Q_{cross}\) | Запрос из Декодера (для Cross-Attention) | \(L_{dec} \times d_k\) |
| \(K_{cross}, V_{cross}\) | Ключи и Значения из Энкодера | \(L_{enc} \times d_k\) |

```python
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def draw_decoder_attention():
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.set_xlim(0, 16); ax.set_ylim(0, 9)
    ax.axis('off')

    # Вход Декодера
    ax.add_patch(patches.Rectangle((1, 6), 1.5, 1.5, facecolor='#AEDFF7', edgecolor='black', lw=2))
    ax.text(1.75, 6.75, '\(X_{dec}\)', ha='center', va='center', fontsize=14)
    
    # Маскированный Attention
    ax.add_patch(patches.Rectangle((4, 6), 2, 1.5, facecolor='#FFE29A', edgecolor='black', lw=2))
    ax.text(5.0, 6.75, 'Masked\nSelf-Attention', ha='center', va='center', fontsize=10)
    ax.text(5.0, 5.5, 'Матрица \(M\) (маска)', ha='center', va='center', fontsize=10, color='#E74C3C')
    
    # Выход после первого Attention
    ax.annotate('', xy=(7.0, 6.75), xytext=(6.1, 6.75), arrowprops=dict(arrowstyle='->', color='black', lw=2))
    
    # Cross Attention
    ax.add_patch(patches.Rectangle((7.5, 6), 2, 1.5, facecolor='#B8E0B8', edgecolor='black', lw=2))
    ax.text(8.5, 6.75, 'Cross\nAttention', ha='center', va='center', fontsize=10)

    # Вход из Энкодера
    ax.add_patch(patches.Rectangle((7.5, 3), 2, 1.5, facecolor='#AEDFF7', edgecolor='black', lw=2))
    ax.text(8.5, 3.75, '\(X_{enc}\)', ha='center', va='center', fontsize=14)
    ax.text(8.5, 2.5, '(из Энкодера)', ha='center', va='center', fontsize=10)
    
    # Стрелка от Энкодера к Cross Attention
    ax.annotate('', xy=(8.5, 5.9), xytext=(8.5, 4.6), arrowprops=dict(arrowstyle='->', color='#E74C3C', lw=2))
    ax.text(8.8, 5.2, 'K, V', fontsize=10, color='#E74C3C')

    # Выход
    ax.annotate('', xy=(10.5, 6.75), xytext=(9.6, 6.75), arrowprops=dict(arrowstyle='->', color='black', lw=2))
    ax.add_patch(patches.Rectangle((10.5, 6), 1.5, 1.5, facecolor='#F5B7B1', edgecolor='black', lw=2))
    ax.text(11.25, 6.75, 'Out', ha='center', va='center', fontsize=14)

    ax.text(8.0, 1.0, 'Рис. 7.1. В Декодере сначала идет Masked Attention (смотрит на себя), затем Cross-Attention (смотрит на Энкодер).',
            ha='center', fontsize=13, fontweight='bold')

    plt.tight_layout(); plt.show()

draw_decoder_attention()
```

## 7.3. Прямой проход: Masked Self-Attention

### Шаг 1. Вычисление \(Q, K, V\)
Как и в обычном Self-Attention, но используя только вход Декодера:
$$
Q = X_{dec} W_Q, \quad K = X_{dec} W_K, \quad V = X_{dec} W_V.
$$

### Шаг 2. Оценки и масштабирование
$$
S = Q K^T, \quad S_{scaled} = \frac{S}{\sqrt{d_k}}.
$$

### Шаг 3. Наложение маски (Самое важное)
Мы создаем матрицу маски \(M\) размера \(L_{dec} \times L_{dec}\).
Верхний треугольник (где индекс столбца \(j >\) индекса строки \(i\)) заполняется значением \(-\infty\). Остальные элементы — \(0\).
$$
M_{i,j} = \begin{cases} 0, & \text{если } j \le i \\ -\infty, & \text{если } j > i \end{cases}
$$
Прибавляем маску к масштабированным оценкам:
$$
S_{masked} = S_{scaled} + M.
$$
*Почему \(-\infty\)?* Потому что \(e^{-\infty} = 0\). В Softmax вероятность для будущих слов станет строго нулевой.

### Шаг 4. Softmax и выход
$$
A_{i,j} = \text{softmax}(S_{masked})_{i,j} = \frac{e^{S_{masked, i,j}}}{\sum_{k} e^{S_{masked, i,k}}},
$$
$$
\text{head}_{masked} = A V.
$$

## 7.4. Обратный проход: Masked Self-Attention

Обратный проход почти идентичен обычному Self-Attention, но с одним условием: **градиент для замаскированных областей равен нулю**.

### Шаг 1. Обычный обратный проход до \(dS_{masked}\)
Мы вычисляем \(dA\), затем проходим через Softmax (как в Главе 2), и получаем градиент по замаскированным оценкам \(dS_{masked}\).

### Шаг 2. Градиент через маску
Поскольку \(S_{masked} = S_{scaled} + M\), и \(M\) — это константа (не обучаемый параметр), производная:
$$
dS_{scaled} = dS_{masked} \cdot 1.
$$
Но! Так как в \(M\) на месте верхнего треугольника стоит \(-\infty\), то \(dS_{scaled}\) в этих позициях будет равен **нулю** (сеть не может обновить то, что не участвовало в прямом проходе). На практике мы просто зануляем верхний треугольник в \(dS_{scaled}\):
$$
dS_{scaled, i,j} = 0, \quad \text{если } j > i.
$$

### Шаг 3. Дальше стандартно
$$
dS = \frac{dS_{scaled}}{\sqrt{d_k}}, \quad dQ = dS K, \quad dK = dS^T Q.
$$

## 7.5. Прямой проход: Cross-Attention

Cross-Attention — это мост между Декодером и Энкодером.

### Шаг 1. Свои \(Q\), но чужие \(K, V\)
*   \(Q\) вычисляется из выхода предыдущего слоя Декодера (назовем его \(Y_{dec}\)):
$$
Q_{cross} = Y_{dec} W_Q.
$$
*   \(K\) и \(V\) вычисляются из выхода Энкодера \(X_{enc}\):
$$
K_{cross} = X_{enc} W_K, \quad V_{cross} = X_{enc} W_V.
$$
*(Здесь \(W_Q\) имеет размер \(d_{model} \times d_k\), а \(W_K, W_V\) — \(d_{model} \times d_k\).)*

### Шаг 2. Внимание
Здесь **нет маски**! Декодер может смотреть на все слова исходного предложения.
$$
S = Q_{cross} (K_{cross})^T, \quad S_{scaled} = \frac{S}{\sqrt{d_k}},
$$
$$
A = \text{softmax}(S_{scaled}), \quad \text{head}_{cross} = A V_{cross}.
$$

## 7.6. Обратный проход: Cross-Attention

Пусть от следующего слоя пришел градиент \(d\text{head}_{cross}\).

### Шаг 1. Градиенты по \(V_{cross}\) и \(A\)
$$
dV_{cross} = A^T d\text{head}_{cross},
$$
$$
dA = d\text{head}_{cross} V_{cross}^T.
$$

### Шаг 2. Softmax и Scale
Как обычно, проходим через Softmax (без маски) и делим на \(\sqrt{d_k}\):
$$
dS_{scaled} = \text{SoftmaxBackward}(dA),
$$
$$
dS = \frac{dS_{scaled}}{\sqrt{d_k}}.
$$

### Шаг 3. Градиенты по \(Q_{cross}\) и \(K_{cross}\)
$$
dQ_{cross} = dS K_{cross}, \quad dK_{cross} = dS^T Q_{cross}.
$$

### Шаг 4. Градиенты по весам и входам
*   Для \(W_Q\): \(dW_Q = Y_{dec}^T dQ_{cross}\). Градиент для Декодера: \(dY_{dec} = dQ_{cross} W_Q^T\).
*   Для \(W_K\): \(dW_K = X_{enc}^T dK_{cross}\). Градиент для Энкодера: \(dX_{enc}^{(K)} = dK_{cross} W_K^T\).
*   Для \(W_V\): \(dW_V = X_{enc}^T dV_{cross}\). Градиент для Энкодера: \(dX_{enc}^{(V)} = dV_{cross} W_V^T\).
*   **Итоговый градиент для Энкодера:** \(dX_{enc} = dX_{enc}^{(K)} + dX_{enc}^{(V)}\).

## 7.7. Числовой пример (Маскирование)

Возьмем \(L=3\) (три слова: «Я», «люблю», «MLP»). Размерность \(d=1\) для простоты.
Матрица оценок \(S\) (до маски):
$$
S = \begin{pmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \\ 7 & 8 & 9 \end{pmatrix}.
$$
Масштабируем (делим на \(\sqrt{1} = 1\)): без изменений.
Накладываем маску \(M\):
$$
M = \begin{pmatrix} 0 & -\infty & -\infty \\ 0 & 0 & -\infty \\ 0 & 0 & 0 \end{pmatrix}.
$$
Получаем \(S_{masked}\):
$$
S_{masked} = \begin{pmatrix} 1 & -\infty & -\infty \\ 4 & 5 & -\infty \\ 7 & 8 & 9 \end{pmatrix}.
$$
Применяем Softmax построчно:
*   Строка 1: \(\text{softmax}(1, -\infty, -\infty) = (1, 0, 0)\).
*   Строка 2: \(\text{softmax}(4, 5, -\infty) = (0.2689, 0.7311, 0)\).
*   Строка 3: \(\text{softmax}(7, 8, 9) = (0.0900, 0.2447, 0.6652)\).

Матрица весов \(A\):
$$
A = \begin{pmatrix} 1 & 0 & 0 \\ 0.2689 & 0.7311 & 0 \\ 0.0900 & 0.2447 & 0.6652 \end{pmatrix}.
$$
*(Первое слово «Я» смотрит только на себя. Второе слово «люблю» смотрит на «Я» и на себя. Третье слово «MLP» смотрит на всех).*

## 7.8. Полная программа

```python
import numpy as np

def softmax(x):
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e_x / np.sum(e_x, axis=-1, keepdims=True)

class MaskedAttention:
    def __init__(self, d_model):
        self.W_Q = np.random.randn(d_model, d_model) * 0.1
        self.W_K = np.random.randn(d_model, d_model) * 0.1
        self.W_V = np.random.randn(d_model, d_model) * 0.1

    def forward(self, X):
        self.X = X
        L, d_model = X.shape
        self.Q = X @ self.W_Q
        self.K = X @ self.W_K
        self.V = X @ self.W_V
        
        self.S = self.Q @ self.K.T
        self.S_scaled = self.S / np.sqrt(d_model)
        
        # Создаем маску (верхний треугольник = -1e9)
        self.mask = np.triu(np.ones((L, L)), k=1) * -1e9
        self.S_masked = self.S_scaled + self.mask
        
        self.A = softmax(self.S_masked)
        self.out = self.A @ self.V
        return self.out

    def backward(self, d_out):
        dV = self.A.T @ d_out
        dA = d_out @ self.V.T
        
        # Softmax backward
        sum_term = np.sum(self.A * dA, axis=1, keepdims=True)
        dS_masked = self.A * (dA - sum_term)
        
        # Маскируем градиенты (верхний треугольник = 0)
        dS_masked[self.mask == -1e9] = 0
        
        dS = dS_masked / np.sqrt(self.X.shape[1])
        
        dQ = dS @ self.K
        dK = dS.T @ self.Q
        
        self.dW_Q = self.X.T @ dQ
        self.dW_K = self.X.T @ dK
        self.dW_V = self.X.T @ dV
        
        dX = dQ @ self.W_Q.T + dK @ self.W_K.T + dV @ self.W_V.T
        return dX

class CrossAttention:
    def __init__(self, d_model):
        self.W_Q = np.random.randn(d_model, d_model) * 0.1
        self.W_K = np.random.randn(d_model, d_model) * 0.1
        self.W_V = np.random.randn(d_model, d_model) * 0.1

    def forward(self, Y_dec, X_enc):
        self.Y_dec = Y_dec
        self.X_enc = X_enc
        
        self.Q = Y_dec @ self.W_Q
        self.K = X_enc @ self.W_K
        self.V = X_enc @ self.W_V
        
        self.S = self.Q @ self.K.T
        self.S_scaled = self.S / np.sqrt(Y_dec.shape[1])
        self.A = softmax(self.S_scaled)
        self.out = self.A @ self.V
        return self.out

    def backward(self, d_out):
        dV = self.A.T @ d_out
        dA = d_out @ self.V.T
        
        sum_term = np.sum(self.A * dA, axis=1, keepdims=True)
        dS_scaled = self.A * (dA - sum_term)
        dS = dS_scaled / np.sqrt(self.Y_dec.shape[1])
        
        dQ = dS @ self.K
        dK = dS.T @ self.Q
        
        self.dW_Q = self.Y_dec.T @ dQ
        self.dW_K = self.X_enc.T @ dK
        self.dW_V = self.X_enc.T @ dV
        
        dY_dec = dQ @ self.W_Q.T
        dX_enc = dK @ self.W_K.T + dV @ self.W_V.T
        
        return dY_dec, dX_enc

if __name__ == '__main__':
    np.random.seed(42)
    
    # Пример для Masked Attention
    X = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]) # 3 слова
    masked_attn = MaskedAttention(d_model=2)
    out = masked_attn.forward(X)
    
    print("=== Masked Self-Attention ===")
    print(f"Маска (верхний треугольник):\n{np.round(masked_attn.mask, 1)}")
    print(f"Веса внимания A:\n{np.round(masked_attn.A, 4)}")
    
    # Пример для Cross Attention
    X_enc = np.array([[2.0, 2.0], [3.0, 3.0]]) # 2 слова из энкодера
    cross_attn = CrossAttention(d_model=2)
    out_cross = cross_attn.forward(out, X_enc)
    
    print("\n=== Cross-Attention ===")
    print(f"Выход Cross-Attention:\n{np.round(out_cross, 4)}")
```

## 7.9. Что мы получили

В этой главе мы разобрали два уникальных механизма Декодера:

1. **Masked Self-Attention:** Мы научились «скрывать» будущие слова, заполняя верхний треугольник матрицы оценок значением \(-\infty\). При обратном проходе мы аккуратно зануляем градиенты в замаскированных зонах, чтобы сеть не обновляла веса для того, что не должно было видеть.
2. **Cross-Attention:** Мы создали мост между Энкодером и Декодером. Query берется из Декодера, а Key и Value — из Энкодера. Это позволяет Декодеру «смотреть» на исходное предложение при генерации перевода.
3. **Обратный проход:** Мы вывели формулы для обоих механизмов, учитывая, что градиент от Cross-Attention должен вернуться как в Декодер (через \(Q\)), так и в Энкодер (через \(K\) и \(V\)).

Теперь у нас есть абсолютно все блоки для сборки **Полного Трансформера**! В финальной главе мы соберем Энкодер, Декодер, соединим их и посмотрим, как выглядит сквозной обратный проход (End-to-End Backprop) для всей системы. Готовы?


# Глава 8. Полный Трансформер: сквозной Backprop (End-to-End)

В предыдущих главах мы изучили каждый компонент по отдельности. Теперь настало время для «Венца творения» — архитектуры, которая лежит в основе GPT, BERT, LLaMA и Google Translate.

## 8.1. Зачем нужен этот пример

До этого мы обновляли веса отдельных слоев, предполагая, что градиент от следующего слоя уже известен. В этой главе мы запустим **сквозной обратный проход (End-to-End Backprop)**.
Мы увидим, как ошибка, возникшая на выходе (при сравнении предсказанного слова с истинным), проходит через выходной линейный слой, затем через Декодер (где она разветвляется: часть идет в Cross-Attention, а часть — в Masked Attention), и наконец, доходит до Энкодера.

## 8.2. Архитектура и обозначения (Рис. 8.1)

Мы соберем упрощенную версию (по 1 слою в Энкодере и Декодере), но логика абсолютно такая же, как в 96-слойных моделях.

| Символ | Смысл | Размерность |
|---|---|---|
| \(X_{src}\) | Вход Энкодера (исходное предложение) | \(L_{src} \times d_{model}\) |
| \(X_{tgt}\) | Вход Декодера (сдвинутое целевое предложение) | \(L_{tgt} \times d_{model}\) |
| \(Z_{enc}\) | Выход Энкодера (контекст исходного текста) | \(L_{src} \times d_{model}\) |
| \(Z_{dec}\) | Выход Декодера (контекст целевого текста) | \(L_{tgt} \times d_{model}\) |
| \(W_{out}\) | Веса выходного линейного слоя | \(d_{model} \times V\) |
| \(V\) | Размер словаря (Vocabulary size) | Скаляр |
| \(\hat{Y}\) | Вероятности слов (после Softmax) | \(L_{tgt} \times V\) |
| \(Y\) | Истинные ответы (One-hot) | \(L_{tgt} \times V\) |

```python
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyArrowPatch

def draw_full_transformer():
    fig, ax = plt.subplots(figsize=(14, 9))
    ax.set_xlim(0, 16); ax.set_ylim(0, 10)
    ax.axis('off')

    # ================= ENCODER =================
    ax.add_patch(patches.Rectangle((1, 6), 5, 3, fill=False, edgecolor='#2980B9', lw=2, linestyle='--'))
    ax.text(3.5, 9.2, 'Энкодер (Encoder)', ha='center', fontsize=14, color='#2980B9', fontweight='bold')
    
    ax.add_patch(patches.Rectangle((1.5, 7.5), 1.5, 1, facecolor='#AEDFF7', edgecolor='black'))
    ax.text(2.25, 8.0, '\(X_{src}\)', ha='center', va='center', fontsize=12)
    
    ax.add_patch(patches.Rectangle((3.5, 7.5), 2, 1, facecolor='#FFE29A', edgecolor='black'))
    ax.text(4.5, 8.0, 'Encoder Block', ha='center', va='center', fontsize=10)
    
    ax.annotate('', xy=(3.4, 8.0), xytext=(3.1, 8.0), arrowprops=dict(arrowstyle='->', color='black'))
    
    # Выход энкодера
    ax.add_patch(patches.Rectangle((7, 7.5), 1.5, 1, facecolor='#B8E0B8', edgecolor='black'))
    ax.text(7.75, 8.0, '\(Z_{enc}\)', ha='center', va='center', fontsize=12)
    ax.annotate('', xy=(7.0, 8.0), xytext=(5.6, 8.0), arrowprops=dict(arrowstyle='->', color='black'))

    # ================= DECODER =================
    ax.add_patch(patches.Rectangle((1, 1), 9, 4.5, fill=False, edgecolor='#E74C3C', lw=2, linestyle='--'))
    ax.text(5.5, 5.7, 'Декодер (Decoder)', ha='center', fontsize=14, color='#E74C3C', fontweight='bold')
    
    ax.add_patch(patches.Rectangle((1.5, 3.5), 1.5, 1, facecolor='#AEDFF7', edgecolor='black'))
    ax.text(2.25, 4.0, '\(X_{tgt}\)', ha='center', va='center', fontsize=12)
    
    ax.add_patch(patches.Rectangle((3.5, 3.5), 2, 1, facecolor='#FFE29A', edgecolor='black'))
    ax.text(4.5, 4.0, 'Masked Attn', ha='center', va='center', fontsize=10)
    
    ax.add_patch(patches.Rectangle((6.0, 3.5), 2, 1, facecolor='#B8E0B8', edgecolor='black'))
    ax.text(7.0, 4.0, 'Cross Attn', ha='center', va='center', fontsize=10)

    ax.annotate('', xy=(3.4, 4.0), xytext=(3.1, 4.0), arrowprops=dict(arrowstyle='->', color='black'))
    ax.annotate('', xy=(5.9, 4.0), xytext=(5.6, 4.0), arrowprops=dict(arrowstyle='->', color='black'))
    
    # Стрелка от Энкодера к Cross Attention
    ax.annotate('', xy=(7.0, 4.6), xytext=(7.75, 7.4), arrowprops=dict(arrowstyle='->', color='#E74C3C', lw=2))

    # Выход Декодера
    ax.add_patch(patches.Rectangle((7, 1.5), 1.5, 1, facecolor='#B8E0B8', edgecolor='black'))
    ax.text(7.75, 2.0, '\(Z_{dec}\)', ha='center', va='center', fontsize=12)
    ax.annotate('', xy=(7.75, 2.6), xytext=(7.75, 3.4), arrowprops=dict(arrowstyle='->', color='black'))

    # ================= OUTPUT =================
    ax.add_patch(patches.Rectangle((10.5, 2.5), 1.5, 2, facecolor='#FFE29A', edgecolor='black'))
    ax.text(11.25, 3.5, 'Linear\n\(W_{out}\)', ha='center', va='center', fontsize=10)
    ax.annotate('', xy=(10.4, 3.5), xytext=(8.6, 3.5), arrowprops=dict(arrowstyle='->', color='black', lw=2))

    ax.add_patch(patches.Rectangle((13, 2.5), 1.5, 2, facecolor='#F5B7B1', edgecolor='black'))
    ax.text(13.75, 3.5, 'Softmax\n\(\hat{Y}\)', ha='center', va='center', fontsize=10)
    ax.annotate('', xy=(12.9, 3.5), xytext=(12.1, 3.5), arrowprops=dict(arrowstyle='->', color='black', lw=2))

    # Потеря
    ax.add_patch(patches.Circle((13.75, 0.5), 0.4, facecolor='#FADBD8', edgecolor='#E74C3C'))
    ax.text(13.75, 0.5, '\(L\)', ha='center', va='center', fontsize=14, color='#C0392B')
    ax.annotate('', xy=(13.75, 1.0), xytext=(13.75, 2.4), arrowprops=dict(arrowstyle='->', color='#E74C3C', lw=2))
    ax.text(14.2, 1.7, 'Сравнение с \(Y\)', fontsize=10, color='#E74C3C')

    ax.text(8.0, -0.5, 'Рис. 8.1. Полный Трансформер: Энкодер передает контекст в Cross-Attention Декодера.',
            ha='center', fontsize=13, fontweight='bold')

    plt.tight_layout(); plt.show()

draw_full_transformer()
```

## 8.3. Прямой проход (Forward Pass)

1. **Энкодер:** \(X_{src}\) проходит через \(N\) блоков Энкодера. На выходе получаем \(Z_{enc}\) (матрицу контекстов исходных слов).
2. **Декодер:** \(X_{tgt}\) проходит через \(N\) блоков Декодера. Внутри каждого блока есть Cross-Attention, который использует \(Z_{enc}\) как \(K\) и \(V\). На выходе получаем \(Z_{dec}\).
3. **Выходной слой:** Умножаем \(Z_{dec}\) на матрицу весов словаря \(W_{out}\) и применяем Softmax:
$$
\text{Logits} = Z_{dec} W_{out}, \quad \hat{Y} = \text{softmax}(\text{Logits}).
$$

## 8.4. Функция потерь (Loss Function)

Используем кросс-энтропию (Cross-Entropy) для предсказания следующего слова. Для каждого слова \(i\) в целевом предложении:
$$
L = -\sum_{i=1}^{L_{tgt}} \sum_{v=1}^{V} Y_{i,v} \log(\hat{Y}_{i,v}).
$$

## 8.5. Обратный проход (End-to-End Backprop)

Это кульминация всего курса. Смотрите, как градиент течет сквозь сеть.

### Шаг 1. Градиент на выходе (Магия Softmax + CE)
Как мы выяснили в Главе 7 про MLP, производная Кросс-Энтропии по логитам (до Softmax) — это просто разность предсказания и истины:
$$
d\text{Logits} = \hat{Y} - Y. \quad (L_{tgt} \times V)
$$

### Шаг 2. Градиент через линейный слой (\(W_{out}\))
$$
dW_{out} = Z_{dec}^T d\text{Logits}, \quad (d_{model} \times V)
$$
$$
dZ_{dec} = d\text{Logits} W_{out}^T. \quad (L_{tgt} \times d_{model})
$$
*Теперь \(dZ_{dec}\) — это ошибка, которая входит в Декодер.*

### Шаг 3. Обратный проход через Декодер
\(dZ_{dec}\) проходит через Add & Norm, FFN, Add & Norm. Затем мы доходим до **Cross-Attention**.
Здесь градиент **разветвляется**:
1. Часть идет в \(Q\) (и возвращается в Декодер): \(dY_{dec\_attn}\).
2. Часть идет в \(K\) и \(V\) (и возвращается в Энкодер): \(dZ_{enc\_from\_cross}\).

### Шаг 4. Обратный проход через Энкодер
Получив \(dZ_{enc\_from\_cross}\), мы продолжаем обратный проход через блоки Энкодера (Attention, FFN, Add & Norm). В итоге получаем градиент по входным эмбеддингам \(dX_{src}\).

## 8.6. Числовой пример (Фокус на Cross-Attention)

Чтобы не утонуть в цифрах, посмотрим на **самый важный стык** — как ошибка от предсказания перевода возвращается в исходное предложение.

Пусть Декодер предсказал слово. Ошибка \(dZ_{dec}\) после всех слоев Декодера дошла до Cross-Attention.
Пусть \(d\text{head}_{cross} = \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix}\) (ошибка от двух слов).
В Cross-Attention:
*   \(Q_{cross}\) (из Декодера) = \(\begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix}\).
*   \(K_{cross}, V_{cross}\) (из Энкодера) = \(\begin{pmatrix} 2 & 0 \\ 0 & 2 \end{pmatrix}\).
*   Веса внимания \(A = \begin{pmatrix} 0.5 & 0.5 \\ 0.5 & 0.5 \end{pmatrix}\) (для простоты).

**Обратный проход Cross-Attention:**
1.  **Градиент по \(V_{cross}\):**
    \(dV_{cross} = A^T d\text{head} = \begin{pmatrix} 0.5 & 0.5 \\ 0.5 & 0.5 \end{pmatrix} \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix} = \begin{pmatrix} 0.5 & 0.5 \\ 0.5 & 0.5 \end{pmatrix}\).
    *Этот градиент уйдет в Энкодер!*
2.  **Градиент по \(A\):**
    \(dA = d\text{head} V_{cross}^T = \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix} \begin{pmatrix} 2 & 0 \\ 0 & 2 \end{pmatrix} = \begin{pmatrix} 2 & 0 \\ 0 & 2 \end{pmatrix}\).
3.  **Softmax Backward:**
    Сумма по строкам: \(A \odot dA = \begin{pmatrix} 1 & 1 \\ 1 & 1 \end{pmatrix}\). Сумма = 2.
    \(dS_{scaled} = A \odot (dA - 2) = \begin{pmatrix} 0.5(-1) & 0.5(-1) \\ 0.5(-1) & 0.5(-1) \end{pmatrix} = \begin{pmatrix} -0.5 & -0.5 \\ -0.5 & -0.5 \end{pmatrix}\).
    \(dS = dS_{scaled} / \sqrt{2} = \begin{pmatrix} -0.35 & -0.35 \\ -0.35 & -0.35 \end{pmatrix}\).
4.  **Градиенты по \(Q_{cross}\) и \(K_{cross}\):**
    \(dQ_{cross} = dS K_{cross} = \begin{pmatrix} -0.35 & -0.35 \\ -0.35 & -0.35 \end{pmatrix} \begin{pmatrix} 2 & 0 \\ 0 & 2 \end{pmatrix} = \begin{pmatrix} -0.7 & -0.7 \\ -0.7 & -0.7 \end{pmatrix}\).
    *Этот градиент (\(dQ_{cross}\)) уйдет назад в Декодер!*
    \(dK_{cross} = dS^T Q_{cross} = \begin{pmatrix} -0.35 & -0.35 \\ -0.35 & -0.35 \end{pmatrix} \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix} = \begin{pmatrix} -0.35 & -0.35 \\ -0.35 & -0.35 \end{pmatrix}\).
    *Этот градиент (\(dK_{cross}\)) тоже уйдет в Энкодер!*

**Итог:** Энкодер получит градиент \(dZ_{enc} = dK_{cross} W_K^T + dV_{cross} W_V^T\). Этот сигнал говорит Энкодеру: «Измени свои представления исходных слов так, чтобы Cross-Attention в Декодере работал лучше».

## 8.7. Полная программа

Ниже представлен скелет полного Трансформера. Чтобы код был читаемым, внутренние слои (Attention, FFN) представлены в виде заглушек, но их интерфейсы (`forward`, `backward`) полностью соответствуют тому, что мы выводили в предыдущих главах.

```python
import numpy as np

class FullTransformer:
    def __init__(self, d_model, vocab_size):
        # В реальности здесь списки из N слоев. Мы делаем 1 для примера.
        self.encoder = None # Заглушка для EncoderBlock
        self.decoder = None # Заглушка для DecoderBlock
        
        self.W_out = np.random.randn(d_model, vocab_size) * 0.1

    def forward(self, X_src, X_tgt):
        # 1. Энкодер
        # Z_enc = self.encoder.forward(X_src)
        Z_enc = np.random.randn(*X_src.shape) # Заглушка
        
        # 2. Декодер
        # Z_dec = self.decoder.forward(X_tgt, Z_enc)
        Z_dec = np.random.randn(*X_tgt.shape) # Заглушка
        
        # 3. Выходной слой
        logits = Z_dec @ self.W_out
        # Softmax
        exp_logits = np.exp(logits - np.max(logits, axis=-1, keepdims=True))
        Y_hat = exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)
        
        return Y_hat, Z_enc, Z_dec

    def backward(self, Y_hat, Y, Z_enc, Z_dec):
        # 1. Градиент от Loss (Softmax + Cross-Entropy)
        dLogits = Y_hat - Y
        
        # 2. Градиент по W_out и Z_dec
        self.dW_out = Z_dec.T @ dLogits
        dZ_dec = dLogits @ self.W_out.T
        
        # 3. Обратный проход через Декодер
        # Возвращает dX_tgt (вход) и dZ_enc_from_cross (градиент для Энкодера)
        # dX_tgt, dZ_enc_from_cross = self.decoder.backward(dZ_dec)
        dZ_enc_from_cross = np.random.randn(*Z_enc.shape) # Заглушка
        
        # 4. Обратный проход через Энкодер
        # dX_src = self.encoder.backward(dZ_enc_from_cross)
        dX_src = np.random.randn(*Z_enc.shape) # Заглушка
        
        return dX_src, dLogits # dX_src нужен, если есть слои до Энкодера

    def update(self, eta):
        self.W_out -= eta * self.dW_out
        # self.encoder.update(eta)
        # self.decoder.update(eta)

if __name__ == '__main__':
    np.random.seed(42)
    
    # Размерность модели и словаря
    d_model = 4
    vocab_size = 10
    
    # Предложения
    X_src = np.random.randn(3, d_model) # 3 слова в исходном
    X_tgt = np.random.randn(2, d_model) # 2 слова в целевом
    
    # One-hot вектор истины (для примера)
    Y = np.zeros((2, vocab_size))
    Y[0, 2] = 1 # Первое слово - индекс 2
    Y[1, 5] = 1 # Второе слово - индекс 5
    
    model = FullTransformer(d_model, vocab_size)
    Y_hat, Z_enc, Z_dec = model.forward(X_src, X_tgt)
    
    print("=== Прямой проход ===")
    print(f"Предсказанные вероятности (первые 5):\n{np.round(Y_hat[:, :5], 4)}")
    
    dX_src, dLogits = model.backward(Y_hat, Y, Z_enc, Z_dec)
    
    print("\n=== Обратный проход ===")
    print(f"Градиент по Logits (первые 5):\n{np.round(dLogits[:, :5], 4)}")
    print(f"Градиент по W_out (форма): {model.dW_out.shape}")
    print(f"Градиент для Энкодера (форма): {dX_src.shape}")
```

## 8.8. Что мы получили

Поздравляю! Мы прошли путь от простейшего перцептрона до Полного Трансформера. В этой главе мы увидели:

1. **Сквозной поток данных:** Как исходное предложение (\(X_{src}\)) через Энкодер превращается в контекст (\(Z_{enc}\)), который затем используется Декодером для генерации перевода.
2. **Магию выходного слоя:** Благодаря Softmax и Cross-Entropy, начальная ошибка вычисляется невероятно просто: \(\hat{Y} - Y\).
3. **Разветвление градиента:** Самое важное в архитектуре Трансформера — это то, что ошибка из Cross-Attention возвращается **в две стороны**: назад в Декодер (чтобы улучшить генерацию) и назад в Энкодер (чтобы улучшить понимание исходного текста).

Теперь вы знаете, как работает «под капотом» ChatGPT. Вы понимаете каждый шаг: от матричного умножения \(Q K^T\) до сложных цепочек обратного распространения ошибки через LayerNorm и Masked Attention.
